<p class="lead">This notebook expands the fourth unit of the course in the R language. The lecture slides are a map; this text is the territory. The unit begins with a <em>data structure</em> — the binary tree, a topic of a first course in algorithms — and ends with one of the most successful learning methods of the last twenty-five years, the random forest. The two halves belong together: a decision tree <em>is</em> a binary tree whose internal nodes ask questions about features and whose leaves hold predictions, and a random forest is a collection of such trees grown on randomised data and averaged. What Sections 2–5 establish about depth, height, traversal and balance is used literally in Sections 6–8, on the breast-cancer data that recurs throughout the course. Every figure and number in the text is produced by the R code you can see and change.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Define a binary tree recursively; use root, leaf, depth, height and degree correctly; count the nodes of a perfect tree of height \(h\).</li>
<li>Implement a binary tree as linked nodes (R environments), as a vector (the \(2i+1\) / \(2i+2\) rule) and as a nested list, and implement the four traversals.</li>
<li>State the binary-search-tree invariant, insert and search in \(O(h)\), and explain why insertion order determines the height.</li>
<li>Explain why balance matters, what a rotation does, and what AVL, red–black and B-trees guarantee.</li>
<li>Describe a decision tree as a binary tree over feature space; derive the Gini, entropy and misclassification impurities; find the best split of a feature by hand and with <code>rpart</code>.</li>
<li>Diagnose overfitting with depth and pruning curves; read a tree's rules and feature importances with the appropriate caveats.</li>
<li>Explain bagging, random feature subsets and the out-of-bag estimate; compare a random forest with a single tree and with the methods of Unit 3 under the methodology of Supplement S1.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read it linearly once, running every code cell, then return by section. Each code cell is short and commented, and the paragraph after it reads its output. Exercises (green boxes) ask you to modify the previous cell and re-run. Blue boxes carry key ideas, orange boxes common misconceptions, and purple boxes connect to other units or to the bioinformatics course (BGM324). Only base R and the packages <code>rpart</code>, <code>rpart.plot</code>, <code>randomForest</code>, <code>mlbench</code>, <code>e1071</code>, <code>class</code> and <code>pROC</code> are used; in a fresh Google Colab R runtime run <code>install.packages(c("rpart.plot","randomForest","mlbench","e1071","pROC"))</code> once. No data is downloaded. If you have taken a data-structures course, read Sections 2–4 quickly but do not skip the worked example of Section 5.</p></div>



In [ ]:
# Setup: packages used throughout. Seeds are fixed so that every number in the text is reproducible.
suppressPackageStartupMessages({
  library(rpart); library(rpart.plot); library(randomForest)
  library(mlbench); library(e1071); library(class); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
cat("R", as.character(getRversion()), "| rpart", as.character(packageVersion("rpart")),
    "| randomForest", as.character(packageVersion("randomForest")),
    "| mlbench", as.character(packageVersion("mlbench")), "\n")



The printout records the versions these notes were built with. `set.seed(0)` fixes the global generator; the stochastic cells below additionally re-seed themselves, so each number is reproducible on its own.

## 1. Why a course on artificial intelligence teaches a data structure

A tree is the natural representation of a <strong>hierarchical decision</strong>. A clinician triaging chest pain asks a first question (is the electrocardiogram abnormal?), the answer determines the next, and so on until a decision is reached. Written down, the protocol is a tree: each question a node, each answer a branch, each decision a leaf. The first computer-derived protocol of that kind, built in 1982 from 482 emergency-room patients by the recursive-partitioning method this unit calls CART, reduced unnecessary coronary-care admissions without missing more infarctions than physicians did.<sup class="c"><a href="#r27">[27]</a></sup>

The same structure appears throughout computer science for a different reason: a tree that halves the remaining possibilities at every node reaches any of \(n\) items in about \(\log_2 n\) steps — twenty steps for a million records. Knuth's treatment of trees and searching remains the reference for the vocabulary and the analysis.<sup class="c"><a href="#r1">[1]</a>,<a href="#r2">[2]</a></sup> The unit proceeds in three steps (Figure 1): the <strong>binary tree</strong> as a data structure (Sections 2–5); the <strong>decision tree</strong> (Section 6), a binary tree whose nodes — thresholds on features — are learned from data and whose depth controls how much it memorises; and the <strong>random forest</strong> (Section 7), hundreds of decision trees grown on random resamples of the patients and random subsets of the features, voting together.

<div class="box key"><h4>The thread of the unit</h4><p>Every property of a binary tree in Sections 2–5 has a consequence for learning. <em>Height</em> is the number of questions asked before a prediction and, in a learned tree, the main control of overfitting. <em>Balance</em> decides whether a prediction costs logarithmic or linear time. <em>Traversal</em> is how a tree is printed as rules, and how <code>rpart</code> stores it: as a data frame indexed by node, the vector representation of Section 2.3.</p></div>



In [ ]:
# Figure 1: the three stages of the unit (binary tree -> decision tree -> forest), drawn with base graphics.
draw_full <- function(labels, fills, main, fs = 0.7) {
  P <- list(c(0,0), c(-1,-1), c(1,-1), c(-1.5,-2), c(-0.5,-2), c(0.5,-2), c(1.5,-2))
  plot.new(); plot.window(xlim = c(-2.3, 2.3), ylim = c(-2.5, 0.5))
  for (i in 2:7) { p <- (i - 2) %/% 2 + 1; segments(P[[p]][1], P[[p]][2], P[[i]][1], P[[i]][2], col = GREY) }
  for (i in 1:7) { points(P[[i]][1], P[[i]][2], pch = 21, bg = fills[i], cex = 4.6); text(P[[i]][1], P[[i]][2], labels[i], cex = fs) }
  title(main = main, cex.main = 0.9)
}
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
draw_full(LETTERS[1:7], rep("white", 7), "1. A binary tree (Sec. 2-5)")
draw_full(c("shape>3?","nuclei>5?","size>4?","benign","malig.","benign","malig."),
          c(rep("#dfe7f1",3), "#e3f1e3","#fbe3d3","#e3f1e3","#fbe3d3"), "2. A decision tree (Sec. 6)", fs = 0.6)
draw_full(c("?","?","?","B","M","M","B"), c(rep("#dfe7f1",3), rep("#e3f1e3",4)), "3. One tree of a forest (Sec. 7)")



<p class="caption"><b>Figure 1.</b> The three objects of the unit: an abstract binary tree (the running example of Section 2); a decision tree over breast-cancer features — the same shape, with a yes/no question at each internal node and a diagnosis at each leaf; and one tree of a forest whose votes are combined.</p>

## 2. Binary trees

### 2.1 Definitions

A <strong>tree</strong> is a set of nodes connected by edges with exactly one path between any two nodes; a <strong>rooted tree</strong> distinguishes one node, the <strong>root</strong>, from which every other node is reached by going "down". The vocabulary, largely due to Knuth, mixes genealogy and botany:<sup class="c"><a href="#r1">[1]</a></sup>

| Term | Definition | In Figure 1 (left) |
|---|---|---|
| **Root** | The unique node with no parent | A |
| **Parent / child** | The node immediately above / below along an edge | B is the parent of D and E |
| **Leaf** | A node with no children | D, E, F, G |
| **Internal node** | A node with at least one child | A, B, C |
| **Degree** of a node | Its number of children | 2 for every internal node |
| **Depth** of a node | Edges from the root; the root has depth 0 | depth(D) = 2 |
| **Height** of a node | Edges on the longest path down to a leaf; a leaf has height 0 | height(B) = 1 |
| **Height** of the tree | Height of the root = maximum depth | 2 |
| **Subtree** rooted at \(v\) | \(v\) and all its descendants | {B, D, E} |

A <strong>binary tree</strong> is a rooted tree in which every node has at most two children, distinguished as <strong>left</strong> and <strong>right</strong>. It is <strong>full</strong> if every node has 0 or 2 children, <strong>perfect</strong> if it is full and all leaves have the same depth (Figure 1, left), and <strong>complete</strong> if every level but the last is full and the last is filled from the left — the shape of heaps and of the vector representation below.

<div class="box warn"><h4>Height is counted in edges; the empty tree has height −1</h4><p>Some textbooks count nodes rather than edges, so a single node has height 1 instead of 0. This notebook and most algorithm texts count edges and give the empty tree height −1, so that the recursive formula below holds without exceptions. An off-by-one turns \(2^{h+1}-1\) into \(2^{h}-1\).</p></div>

### 2.2 The recursive definition

<em>A binary tree is either empty, or a root node together with two binary trees, its left and right subtrees.</em> Every algorithm on trees follows this shape — handle the root, recurse left, recurse right — and every property is proved by induction on the height. For instance

\[ h(T) = \begin{cases} -1 & T \text{ empty} \\ 1 + \max\big(h(T_\text{left}), h(T_\text{right})\big) & \text{otherwise,} \end{cases} \]

and \(n(T) = 0\) for the empty tree, \(1 + n(T_\text{left}) + n(T_\text{right})\) otherwise.

### 2.3 Three representations

The first representation is <strong>linked nodes</strong>. R's ordinary objects are copied when assigned, so to obtain the reference semantics of a linked structure we store each node in an <strong>environment</strong>: a mutable container with a value and two references, <code>left</code> and <code>right</code>, <code>NULL</code> when a child is absent. The cell builds the tree of Figure 1 this way and computes its height and size with the recursive definitions.



In [ ]:
# Representation 1: linked nodes as environments (reference semantics, like a pointer-based tree).
node <- function(v, l = NULL, r = NULL) { e <- new.env(); e$val <- v; e$left <- l; e$right <- r; e }

tree_height <- function(t) {              # height in edges, iterative (safe for the deep trees of Sec. 3-4)
  if (is.null(t)) return(-1L)
  st <- list(list(n = t, d = 0L)); h <- -1L
  while (length(st)) {
    x <- st[[length(st)]]; st[[length(st)]] <- NULL; h <- max(h, x$d)
    if (!is.null(x$n$left))  st[[length(st) + 1]] <- list(n = x$n$left,  d = x$d + 1L)
    if (!is.null(x$n$right)) st[[length(st) + 1]] <- list(n = x$n$right, d = x$d + 1L)
  }
  h
}
tree_count  <- function(t) if (is.null(t)) 0L else 1L + tree_count(t$left) + tree_count(t$right)
tree_leaves <- function(t) if (is.null(t)) c() else if (is.null(t$left) && is.null(t$right)) t$val else c(tree_leaves(t$left), tree_leaves(t$right))

# The seven-node tree of Figure 1 (left), built bottom-up: leaves first, then parents, then the root.
T <- node("A", node("B", node("D"), node("E")), node("C", node("F"), node("G")))
cat("root:", T$val, "| left child of root:", T$left$val, "| right child of B:", T$left$right$val, "\n")
cat("height:", tree_height(T), "| nodes:", tree_count(T), "| leaves:", paste(tree_leaves(T), collapse = " "), "\n")
cat("height of subtree at B:", tree_height(T$left), "| of a leaf:", tree_height(T$left$left), "| of the empty tree:", tree_height(NULL), "\n")



The functions are the recursive definition as code, and the outputs confirm the hand count: height 2, seven nodes, four leaves, height(leaf) = 0, height(empty) = −1.

The second representation is a <strong>vector</strong>: number the nodes level by level, root at 0; the children of node \(i\) are at \(2i+1\) and \(2i+2\) and its parent at \(\lfloor (i-1)/2 \rfloor\). No references are stored — the shape is in the arithmetic. It is compact for complete trees (the binary heap is exactly this<sup class="c"><a href="#r2">[2]</a></sup>) and wasteful otherwise, because missing nodes still occupy slots. The third is a <strong>nested list</strong>, convenient for serialising to JSON.



In [ ]:
# Representation 2: a level-order vector (children of i at 2i+1, 2i+2, 0-based). Representation 3: a nested list.
to_vector <- function(t) {
  arr <- rep(NA_character_, 2 ^ (tree_height(t) + 1) - 1)
  fill <- function(nd, i) { if (is.null(nd)) return(); arr[i + 1] <<- nd$val; fill(nd$left, 2 * i + 1); fill(nd$right, 2 * i + 2) }
  fill(t, 0); arr
}
to_list <- function(t) if (is.null(t)) NULL else list(value = t$val, left = to_list(t$left), right = to_list(t$right))

arr <- to_vector(T)
cat("vector (0-based index shown):", paste0(0:(length(arr) - 1), ":", arr, collapse = "  "), "\n")
for (i in 0:2) cat(sprintf("  node %s at index %d: children at %d (%s) and %d (%s); parent of %d is %d (%s)\n",
    arr[i + 1], i, 2 * i + 1, arr[2 * i + 2], 2 * i + 2, arr[2 * i + 3], 2 * i + 1, (2 * i) %/% 2, arr[(2 * i) %/% 2 + 1]))
cat("nested list, left spine:", to_list(T)$value, "->", to_list(T)$left$value, "->", to_list(T)$left$left$value, "\n")



The vector reads `A B C D E F G`, the level-order listing, and the arithmetic recovers every parent–child relation. Section 6.1 shows that <code>rpart</code> stores a learned tree in exactly this spirit — a data frame whose row names are node numbers obeying the same rule (children of node \(i\) at \(2i\) and \(2i+1\), one-based), so that unbalanced trees are stored without wasted slots.

### 2.4 Traversals

To <strong>traverse</strong> a tree is to visit every node once in a defined order. The recursive definition offers three places to visit the root — before, between or after the recursive calls — plus one breadth-first order:<sup class="c"><a href="#r1">[1]</a></sup>

| Traversal | Order | Typical use |
|---|---|---|
| **Pre-order** | root, left, right | printing a tree as nested rules (Section 6) |
| **In-order** | left, root, right | on a search tree, yields the keys in sorted order (Section 3) |
| **Post-order** | left, right, root | deleting a tree (children first); evaluating expression trees |
| **Level-order** | depth 0, then depth 1, … | the vector representation |

By hand on Figure 1: pre-order visits A, then B's subtree (B, D, E), then C's: `A B D E C F G`; in-order `D B E A F C G`; post-order `D E B F G C A`; level-order `A B C D E F G`.



In [ ]:
# The four traversals. The three depth-first ones differ only in WHERE the root is visited.
preorder  <- function(t) if (is.null(t)) c() else c(t$val, preorder(t$left),  preorder(t$right))
inorder   <- function(t) if (is.null(t)) c() else c(inorder(t$left),  t$val, inorder(t$right))
postorder <- function(t) if (is.null(t)) c() else c(postorder(t$left), postorder(t$right), t$val)
levelorder <- function(t) {                      # breadth-first: a queue of discovered-but-unvisited nodes
  if (is.null(t)) return(c())
  out <- c(); q <- list(t)
  while (length(q)) { nd <- q[[1]]; q <- q[-1]; out <- c(out, nd$val)
    if (!is.null(nd$left))  q <- c(q, list(nd$left))
    if (!is.null(nd$right)) q <- c(q, list(nd$right)) }
  out
}
for (nm in c("pre-order", "in-order", "post-order", "level-order"))
  cat(sprintf("%-12s %s\n", nm, paste(switch(nm, "pre-order" = preorder, "in-order" = inorder,
      "post-order" = postorder, levelorder)(T), collapse = " ")))



All four match the hand trace, and only the position of `t$val` separates the three depth-first traversals. In-order is the important one: on a search tree it yields the sorted keys.

### 2.5 Counting nodes

A <strong>perfect</strong> tree of height \(h\) has \(2^d\) nodes at depth \(d\), so

\[ n = \sum_{d=0}^{h} 2^d = 2^{h+1} - 1, \qquad h = \log_2(n+1) - 1, \]

with \(2^h\) leaves. Conversely, any binary tree with \(n\) nodes has height at least \(\lceil \log_2(n+1) \rceil - 1\) and at most \(n-1\) (a chain). These two bounds, logarithmic and linear, are the whole story of Sections 3 and 4.



In [ ]:
# Perfect binary trees of height 0..6: count nodes and leaves and compare with 2^(h+1)-1 and 2^h.
perfect <- function(h, label = 1) if (h < 0) NULL else node(label, perfect(h - 1, 2 * label), perfect(h - 1, 2 * label + 1))
tab <- do.call(rbind, lapply(0:6, function(h) {
  t <- perfect(h)
  data.frame(height_h = h, nodes = tree_count(t), fml_2h1_minus1 = 2 ^ (h + 1) - 1,
             leaves = length(tree_leaves(t)), fml_2h = 2 ^ h, height_recomputed = tree_height(t))
}))
print(tab, row.names = FALSE)



The counts agree with the formula: 1, 3, 7, 15, 31, 63, 127. Read the other way, a perfect tree holding a million items has height only 19, so any question about its contents is answered in about twenty steps — <em>provided the tree stays close to perfect</em>.

<div class="exercise"><h4>Exercise 1</h4><p>Modify <code>perfect()</code> so that each node has only a <em>right</em> child. How many nodes does a chain of height \(h\) have, and how many slots does <code>to_vector()</code> waste for a chain of height 4? This is the degenerate shape of Section 3.4.</p></div>

## 3. Binary search trees

### 3.1 The invariant

A <strong>binary search tree</strong> (BST) holds <em>keys</em> from an ordered set and satisfies, at every node \(v\), the <strong>BST invariant</strong>:

\[ \text{every key in the left subtree of } v \;<\; \text{key}(v) \;<\; \text{every key in the right subtree of } v , \]

a condition on whole subtrees, not only on the two children. Duplicate keys are disallowed here.

To <strong>search</strong> for \(k\): start at the root; stop if \(k\) equals the node's key; go left if smaller, right if larger; if the child is absent, \(k\) is not in the tree. To <strong>insert</strong> \(k\): search, and attach a new leaf where the search fell off. Each step descends one level, so both operations cost at most the height: \(O(h)\). What matters is how large \(h\) is relative to \(n\).<sup class="c"><a href="#r2">[2]</a></sup> The implementation stores the tree in a holder environment and counts key comparisons for the cost experiments of Sections 3.4 and 4.5; insertion and search are written as loops (not recursion) so they are safe on the deep trees to come.



In [ ]:
# A binary search tree on distinct keys; counts comparisons; insert and search are iterative.
new_bst <- function() { b <- new.env(); b$root <- NULL; b$comparisons <- 0L; b }
bst_insert <- function(b, key) {
  if (is.null(b$root)) { b$root <- node(key); return(invisible()) }
  cur <- b$root
  repeat {
    b$comparisons <- b$comparisons + 1L
    if (key < cur$val)      { if (is.null(cur$left))  { cur$left  <- node(key); break }; cur <- cur$left }
    else if (key > cur$val) { if (is.null(cur$right)) { cur$right <- node(key); break }; cur <- cur$right }
    else break                                           # equal keys are ignored
  }
  invisible()
}
bst_search <- function(b, key) {                         # returns the path of keys visited
  path <- c(); cur <- b$root
  while (!is.null(cur)) { path <- c(path, cur$val); b$comparisons <- b$comparisons + 1L
    if (key == cur$val) return(list(path = path, found = TRUE))
    cur <- if (key < cur$val) cur$left else cur$right }
  list(path = path, found = FALSE)
}
bst <- new_bst()
for (k in c(41, 20, 65, 11, 29, 50, 91, 32, 72, 99)) bst_insert(bst, k)
cat("in-order traversal:", paste(inorder(bst$root), collapse = " "), "\n")
cat("height:", tree_height(bst$root), "| nodes:", tree_count(bst$root), "\n")
for (k in c(32, 72, 60)) { s <- bst_search(bst, k)
  cat(sprintf("search %d: path %s -> %s after %d comparisons\n", k, paste(s$path, collapse = " "),
      if (s$found) "found" else "absent", length(s$path))) }



The in-order traversal returns the keys in ascending order although they were inserted in no particular order — the invariant made visible, and the reason <em>tree sort</em> works. The ten keys produced height 3, the minimum \(\lceil \log_2 11 \rceil - 1\), and each search stops after at most \(h+1 = 4\) comparisons, found or not.

### 3.2 Why in-order yields sorted keys

By induction on the height: in-order visits the left subtree, the root, then the right subtree; by the invariant every left key is smaller than the root and every right key larger; by the inductive hypothesis each subtree is visited in sorted order; the concatenation is sorted. The same argument gives an \(O(n)\) check of the invariant: traverse in-order and verify the sequence increases.

### 3.3 Deletion, in outline

A <em>leaf</em> is removed; a node with <em>one child</em> is replaced by that child; a node with <em>two children</em> receives the key of its <strong>in-order successor</strong> (the smallest key of its right subtree), and the successor, which has at most one child, is deleted instead. The invariant is preserved and the cost is \(O(h)\). Knuth notes that repeated deletions with this rule slowly unbalance the tree — one motivation for Section 4.<sup class="c"><a href="#r2">[2]</a></sup>

### 3.4 Degeneration: the height depends on the insertion order

Nothing in the insertion algorithm keeps the tree short. If keys arrive in sorted order each new key goes right: the tree is a chain of height \(n-1\) and search is a linear scan dressed as a tree. If keys arrive in random order, the root splits the rest roughly in half, each subtree is again a random BST, and the expected height is proportional to \(\log n\).<sup class="c"><a href="#r2">[2]</a></sup> Sorted input is not contrived: patient identifiers, timestamps and accession numbers are generated in increasing order.



In [ ]:
# Height of a BST after inserting n keys in sorted order (degenerate) versus random order (20 shuffles per n).
set.seed(0)
ns <- c(10, 20, 50, 100, 200, 500, 1000, 2000)
res <- do.call(rbind, lapply(ns, function(n) {
  st <- new_bst(); for (k in 0:(n - 1)) bst_insert(st, k)          # sorted input -> chain
  rh <- replicate(20, { t <- new_bst(); for (k in sample(0:(n - 1))) bst_insert(t, k); tree_height(t$root) })
  data.frame(n = n, sorted = tree_height(st$root), random_mean = mean(rh), random_max = max(rh), log2n = log2(n))
}))
plot(res$n, res$sorted, log = "xy", type = "b", pch = 15, col = ORANGE, xlab = "number of keys n",
     ylab = "height of the tree", main = "BST height versus number of keys and insertion order")
lines(res$n, res$random_mean, type = "b", pch = 19, col = BLUE)
lines(res$n, res$log2n, lty = 2, col = GREY)
legend("topleft", bty = "n", cex = 0.8, col = c(ORANGE, BLUE, GREY), pch = c(15, 19, NA), lty = c(1, 1, 2),
       legend = c("sorted insertion: h = n - 1", "random insertion: mean of 20 trees", "log2(n), perfect-tree bound"))
print(round(res, 1), row.names = FALSE)



<p class="caption"><b>Figure 2.</b> Height of a binary search tree versus number of keys, logarithmic axes. Sorted insertion gives \(h = n-1\); random insertion gives heights growing like a constant times \(\log_2 n\). Dashed: the perfect-tree lower bound.</p>

The random trees are two to three times taller than a perfect tree — the price of not controlling the shape — but still logarithmic, whereas the chain is linear: some two dozen comparisons per search at \(n = 2{,}000\) where the chain needs up to 2,000. A BST is only as fast as its shape, and the shape is at the mercy of the data unless the algorithm maintains it.

<div class="exercise"><h4>Exercise 2</h4><p>Insert <code>0:(n-1)</code> after swapping 5 % of adjacent pairs at random: is the height closer to the sorted or the random case? Then insert in "median first, then the medians of the two halves, …" order. What height results, and why is it the minimum possible?</p></div>

## 4. Balanced trees

### 4.1 Why balance matters and what a rotation does

A search tree is <strong>balanced</strong> when its height is \(O(\log n)\) for every sequence of insertions and deletions, and <em>self-balancing</em> when it restores balance after each operation with a small, local amount of work, so that every operation is logarithmic in the worst case rather than on average. The restoring tool is the <strong>rotation</strong>: a rearrangement of a node and one child that changes the heights of the two sides by one while preserving the in-order sequence, and therefore the invariant. In Figure 3, a right rotation about \(y\) with left child \(x\) makes \(x\) the parent and \(y\) its right child, and moves subtree \(B\) (keys between \(x\) and \(y\)) from the right of \(x\) to the left of \(y\), at the cost of a constant number of reference changes.



In [ ]:
# Figure 3: a right rotation about y. In-order A x B y C is preserved; the heights of the sides change by one.
draw_rot <- function(top, child, side, main) {
  tx <- 0; ty <- 1; cx <- if (side == "left") -1 else 1; cy <- 0
  plot.new(); plot.window(xlim = c(-2.3, 2.3), ylim = c(-1.6, 1.6)); segments(tx, ty, cx, cy, col = GREY)
  tri <- function(ax, ay, lab) { polygon(c(ax - 0.35, ax + 0.35, ax), c(ay - 0.4, ay - 0.4, ay + 0.4), col = "#e3f1e3"); text(ax, ay - 0.1, lab, cex = 0.8) }
  if (side == "left") { segments(cx, cy, cx - 0.7, -1, col = GREY); segments(cx, cy, cx + 0.3, -1, col = GREY); segments(tx, ty, 1.2, -0.2, col = GREY)
    tri(cx - 0.7, -1, "A"); tri(cx + 0.3, -1, "B"); tri(1.2, -0.6, "C") }
  else { segments(tx, ty, -1.2, -0.2, col = GREY); segments(cx, cy, cx - 0.3, -1, col = GREY); segments(cx, cy, cx + 0.7, -1, col = GREY)
    tri(-1.2, -0.6, "A"); tri(cx - 0.3, -1, "B"); tri(cx + 0.7, -1, "C") }
  points(c(tx, cx), c(ty, cy), pch = 21, bg = "#dfe7f1", cex = 3.6); text(c(tx, cx), c(ty, cy), c(top, child), cex = 0.9)
  title(main = main, cex.main = 0.85)
}
par(mfrow = c(1, 2), mar = c(1, 1, 2, 1))
draw_rot("y", "x", "left", "before: y is the root, left-heavy")
draw_rot("x", "y", "right", "after a right rotation about y: x is the root")
mtext("in-order unchanged:  A  x  B  y  C", side = 1, outer = TRUE, line = -1.4, cex = 0.8, font = 3)



<p class="caption"><b>Figure 3.</b> A right rotation about \(y\): subtree \(A\) rises one level, \(C\) sinks one, \(B\) changes parent, and the in-order sequence \(A, x, B, y, C\) is unchanged. The left rotation is the mirror image; a double rotation handles the case in which the heavy subtree is \(B\).</p>

### 4.2 AVL trees

The first self-balancing tree, published in 1962 by Adelson-Velskii and Landis, maintains the <strong>AVL condition</strong>: at every node the heights of the two subtrees differ by at most one.<sup class="c"><a href="#r3">[3]</a></sup> After an insertion the algorithm walks back up the path updating a stored <em>balance factor</em>; at the first node where it reaches \(\pm 2\), one single or double rotation restores the condition. The height is at most about \(1.44\log_2(n+2)\), so every operation is \(O(\log n)\) in the worst case; Knuth derives the bound via Fibonacci numbers.<sup class="c"><a href="#r2">[2]</a></sup>

### 4.3 Red–black trees

A <strong>red–black tree</strong> relaxes the AVL condition in exchange for fewer rotations: each node carries a colour bit, the root is black, a red node has no red child, and every path from a node to the empty leaves below passes the same number of black nodes. No root-to-leaf path is then more than twice as long as any other, so the height is at most \(2\log_2(n+1)\). Bayer introduced the structure in 1972 as the <em>symmetric binary B-tree</em><sup class="c"><a href="#r4">[4]</a></sup> and Guibas and Sedgewick gave it the colour formulation in 1978.<sup class="c"><a href="#r5">[5]</a></sup> It is the usual implementation of ordered maps (C++ <code>std::map</code>, Java <code>TreeMap</code>); R's environments and named lists are hash tables, faster for exact lookup but unable to answer range queries.

### 4.4 B-trees in databases

On disk the cost that matters is the number of blocks read, and a binary node wastes a block on one key. The <strong>B-tree</strong> of Bayer and McCreight stores hundreds of keys per node, sized to a disk page, with all leaves at the same depth and all nodes at least half full.<sup class="c"><a href="#r6">[6]</a></sup> With branching factor 500, 125 million keys fit in height 3. Every relational database index uses a B-tree variant; when a hospital system retrieves a record by identifier in milliseconds, it is walking one.

### 4.5 Experiment: search cost in a degenerate versus a balanced tree

Implementing AVL rebalancing is beyond the unit, but its <em>effect</em> is easy to show: the same keys inserted in sorted order give a chain, and inserted median-first (the solution of Exercise 2) give a perfect tree. The cell inserts 1,023 keys both ways and counts the comparisons of 2,000 random searches.



In [ ]:
# Search cost: 1,023 keys inserted as a chain (sorted) versus as a perfect tree (median-first), 2,000 searches each.
median_first <- function(keys) {                 # insertion order that builds a perfect BST from a sorted vector
  if (!length(keys)) return(integer(0))
  m <- (length(keys) + 1) %/% 2
  c(keys[m], median_first(keys[seq_len(m - 1)]), median_first(keys[(m + 1):length(keys)][seq_len(length(keys) - m)]))
}
keys <- 0:1022                                    # 1023 = 2^10 - 1 -> a perfect tree of height 9 is possible
chain <- new_bst(); for (k in keys)                bst_insert(chain, k)
balanced <- new_bst(); for (k in median_first(keys)) bst_insert(balanced, k)
set.seed(1); queries <- sample(0:1022, 2000, replace = TRUE)
out <- do.call(rbind, lapply(list(c("degenerate (sorted insertion)", "chain"), c("balanced (median-first)", "balanced")),
  function(p) { t <- get(p[2]); t$comparisons <- 0L; for (q in queries) bst_search(t, q)
    data.frame(tree = p[1], height = tree_height(t$root), comparisons_per_search = round(t$comparisons / length(queries), 1),
               worst_case_h_plus_1 = tree_height(t$root) + 1) }))
print(out, row.names = FALSE)



The balanced tree has height 9, the minimum for 1,023 keys, and answers a search in about nine comparisons on average; the chain has height 1,022 and needs some five hundred for the same keys. A self-balancing tree pays a small constant per insertion to keep the second column logarithmic. In the second half of the unit the trees are shaped by data and depth limits rather than by rotations, but the cost of a prediction is still the depth of the leaf reached, and a deep tree is still a slow and — as we shall see — an over-specific one.

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>A <em>suffix tree</em> indexes a genome so that any substring is found in time proportional to its length; the <em>k-d tree</em> and ball tree speed up the nearest-neighbour search of Unit 3; the <em>phylogenetic tree</em> of UPGMA or neighbour-joining is a binary tree whose post-order traversal computes ancestral states; the clustering dendrogram of Unit 5 is a binary tree built from the leaves up.</p></div>

## 5. Worked example by hand: building a search tree

Insert <strong>50, 30, 70, 20, 40, 60, 80</strong>, in this order, into an empty BST ("smaller goes left, larger goes right"):

| Step | Key | Comparisons | Placement |
|---|---|---|---|
| 1 | 50 | — | root |
| 2 | 30 | 30 < 50 → left | left child of 50 |
| 3 | 70 | 70 > 50 → right | right child of 50 |
| 4 | 20 | 20 < 50 → left; 20 < 30 → left | left child of 30 |
| 5 | 40 | 40 < 50 → left; 40 > 30 → right | right child of 30 |
| 6 | 60 | 60 > 50 → right; 60 < 70 → left | left child of 70 |
| 7 | 80 | 80 > 50 → right; 80 > 70 → right | right child of 70 |

```
            50
          /    \
        30      70
       /  \    /  \
     20   40  60   80
```

The tree has 7 nodes and height 2 and is perfect — this order is the median-first order of Section 4.5. Its traversals: **pre-order** 50, 30, 20, 40, 70, 60, 80; **in-order** 20, 30, 40, 50, 60, 70, 80 (sorted, as the invariant requires); **post-order** 20, 40, 30, 60, 80, 70, 50; **level-order** 50, 30, 70, 20, 40, 60, 80. Searching for 60 follows 50 → 70 → 60 (three comparisons); searching for 65 follows the same path, finds no right child of 60, and reports absence after three comparisons. The same keys inserted in sorted order give a chain of height 6 in which the search for 80 takes seven comparisons. The cell confirms every statement and draws both trees.



In [ ]:
# Confirming the hand-worked example of Section 5 in code, then drawing both trees from the structure.
hand <- new_bst(); for (k in c(50, 30, 70, 20, 40, 60, 80)) bst_insert(hand, k)
cat("nodes:", tree_count(hand$root), "| height:", tree_height(hand$root), "| leaves:", paste(tree_leaves(hand$root), collapse = " "), "\n")
for (nm in c("pre-order", "in-order", "post-order", "level-order"))
  cat(sprintf("%-12s %s\n", nm, paste(switch(nm, "pre-order" = preorder, "in-order" = inorder, "post-order" = postorder, levelorder)(hand$root), collapse = " ")))
for (q in c(60, 65)) { s <- bst_search(hand, q); cat(sprintf("search %d: %s (%d comparisons)\n", q, if (s$found) "found" else "absent", length(s$path))) }
chain7 <- new_bst(); for (k in sort(c(50, 30, 70, 20, 40, 60, 80))) bst_insert(chain7, k)
cat("same keys inserted sorted -> height", tree_height(chain7$root), "| search 80 takes", length(bst_search(chain7, 80)$path), "comparisons\n")

draw_bst <- function(root, main) {
  ord <- inorder(root); pos <- new.env(); hgt <- tree_height(root)
  place <- function(nd, d) { if (is.null(nd)) return(); x <- match(nd$val, ord); assign(as.character(nd$val), c(x, -d), pos)
    for (ch in list(nd$left, nd$right)) if (!is.null(ch)) segments(x, -d, match(ch$val, ord), -d - 1, col = GREY)
    place(nd$left, d + 1); place(nd$right, d + 1) }
  plot.new(); plot.window(xlim = c(0.5, length(ord) + 0.5), ylim = c(-hgt - 0.6, 0.6)); place(root, 0)
  for (k in ord) { p <- get(as.character(k), pos); points(p[1], p[2], pch = 21, bg = "#dfe7f1", cex = 2.8); text(p[1], p[2], k, cex = 0.7) }
  title(main = main, cex.main = 0.85)
}
par(mfrow = c(1, 2), mar = c(1, 1, 2, 1))
draw_bst(hand$root, "inserted 50,30,70,20,40,60,80 (height 2)")
draw_bst(chain7$root, "same keys inserted sorted (height 6)")



<p class="caption"><b>Figure 4.</b> The two trees of the worked example, drawn from the data structure: horizontal position is the in-order rank, vertical position the depth. Same keys, same in-order sequence, heights 2 and 6.</p>

The code reproduces the hand trace exactly; you should be able to perform it for any short list of keys on paper.

## 6. From binary trees to decision trees

### 6.1 A decision tree is a binary tree over feature space

The recurring dataset of the course is the <strong>Wisconsin breast-cancer</strong> data. The <code>mlbench</code> package ships the original version of Wolberg and collaborators — 699 fine-needle aspirates, nine cytological features each scored by a pathologist from 1 to 10, labelled benign or malignant.<sup class="c"><a href="#r28">[28]</a>,<a href="#r4">[4]</a></sup> It is the R stand-in for the dataset used throughout the Python edition. A <strong>decision tree</strong> classifier is a binary tree in which every internal node holds a test "is feature \(j\) ≤ threshold \(t\)?", the left child is taken when the answer is yes, and every leaf holds a class. Classifying a patient is a search in the sense of Section 3, driven by the patient's features, in which the path itself is the answer. Geometrically each test is a hyperplane perpendicular to one axis, so the leaves partition feature space into axis-aligned boxes. The cell loads and curates the data, fits a depth-2 tree to the training part of a 70/30 split (used for the rest of the unit, following Supplement S1), draws it, and inspects <code>rpart</code>'s internal representation.



In [ ]:
# Load and curate the data; a depth-2 tree; its internal data-frame representation.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # drop the Id column
bc <- bc[complete.cases(bc), ]                             # drop the 16 rows with missing values
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # ordinal -> numeric
set.seed(1)
idx <- sample(nrow(bc), round(0.7 * nrow(bc)))
tr <- bc[idx, ]; te <- bc[-idx, ]
cat(sprintf("%d complete cases; training %d, test %d; classes: %s\n", nrow(bc), nrow(tr), nrow(te),
            paste(names(table(bc$Class)), table(bc$Class), collapse = ", ")))

ctl <- function(d) rpart.control(maxdepth = d, cp = 0, minsplit = 2, minbucket = 1, xval = 0)
set.seed(1); tree2 <- rpart(Class ~ ., tr, method = "class", control = ctl(2))
rpart.plot(tree2, type = 2, extra = 104, box.palette = list("#fbe3d3", "#dfe7f1"), main = "A depth-2 tree (training part)")
print(tree2$frame[, c("var", "n", "dev", "yval")])
cat("node numbers (row names):", paste(rownames(tree2$frame), collapse = " "), "\n")



<p class="caption"><b>Figure 5.</b> A depth-2 tree learned from 478 training cases. Each box shows the majority class, the class proportions and the share of training cases reaching the node; an internal node also shows its test. A root-to-leaf path is a rule: the first question concerns cell shape, the next cell size or the appearance of the bare nuclei.</p>

The printed data frame proves the claim that opened the section: <code>rpart</code> stores the tree as a table indexed by node, with the special variable <code>&lt;leaf&gt;</code> marking a leaf. The tree has 7 nodes, the perfect-tree count of Section 2.5, and the row names — the node numbers — are 1, 2, 4, 5, 3, 6, 7: exactly the vector rule of Section 2.3, the children of node \(i\) sitting at \(2i\) and \(2i+1\). Everything <code>rpart.plot</code> and the rule text print is a traversal of this table.

### 6.2 Impurity: how good is a split?

Learning a tree means choosing the tests. The idea common to the two traditions that invented the method — the statistical CART of Breiman, Friedman, Olshen and Stone<sup class="c"><a href="#r7">[7]</a></sup> and Quinlan's ID3 and C4.5<sup class="c"><a href="#r8">[8]</a>,<a href="#r9">[9]</a></sup> — is that a good test sends the two classes to different sides. The <strong>impurity</strong> of a node makes this numerical: 0 when all cases share one class, largest when the classes are equally mixed. For class proportions \(p_1, \dots, p_K\):

\[ \text{Gini: } G = 1 - \sum_k p_k^2, \qquad \text{Entropy: } H = -\sum_k p_k \log_2 p_k, \qquad \text{Misclassification: } E = 1 - \max_k p_k . \]

Gini is the probability that two cases drawn from the node belong to different classes (CART's default, and <code>rpart</code>'s); entropy is the expected number of bits to encode a case's class (ID3); misclassification is the error of the majority prediction. For two classes, \(G = 2p(1-p)\), \(H = -p\log_2 p - (1-p)\log_2(1-p)\), \(E = \min(p, 1-p)\).

The <strong>quality of a split</strong> is the decrease in impurity, weighted by the number of cases on each side:

\[ \Delta I = I - \left( \frac{n_L}{n} I_L + \frac{n_R}{n} I_R \right), \]

called the <strong>information gain</strong> when \(I\) is the entropy. The chosen split maximises \(\Delta I\).



In [ ]:
# The three impurity measures for a two-class node as a function of the proportion p of one class.
p <- seq(0.001, 0.999, length.out = 400)
gini <- 2 * p * (1 - p)
entropy <- -(p * log2(p) + (1 - p) * log2(1 - p))
miscl <- pmin(p, 1 - p)
plot(p, entropy, type = "l", col = BLUE, lwd = 2, ylim = c(0, 1.05),
     xlab = "proportion p of one class at the node", ylab = "impurity", main = "Impurity of a two-class node")
lines(p, gini, col = ORANGE, lwd = 2); lines(p, miscl, col = GREEN, lwd = 2); lines(p, entropy / 2, col = BLUE, lty = 3)
legend("topright", bty = "n", cex = 0.8, lwd = 2, col = c(BLUE, ORANGE, GREEN, BLUE), lty = c(1, 1, 1, 3),
       legend = c("entropy", "Gini = 2p(1-p)", "misclassification = min(p,1-p)", "entropy / 2"))
cat(sprintf("at p=0.5: Gini %.3f, entropy %.3f, misclassification %.3f\n", 0.5, 1.0, 0.5))
cat(sprintf("at p=0.9: Gini %.3f, entropy %.3f, misclassification %.3f\n", 2 * 0.9 * 0.1, -(0.9 * log2(0.9) + 0.1 * log2(0.1)), 0.1))



<p class="caption"><b>Figure 6.</b> The three impurity measures for a two-class node: all vanish for a pure node and peak at \(p = 0.5\). Gini and entropy are strictly concave and nearly the same shape (dotted: entropy rescaled); misclassification is piecewise linear.</p>

The shape matters more than the values. Because Gini and entropy are <em>strictly concave</em>, any split that makes the children less mixed decreases them even if no child changes its majority class; the misclassification error can stay unchanged under a useful split when both children keep the parent's majority class (a 60 : 40 parent split into 30 : 10 and 30 : 30 keeps \(E = 0.4 \times 0.25 + 0.6 \times 0.5 = 0.40\), the parent's value, while Gini falls from 0.48 to \(0.4 \times 0.375 + 0.6 \times 0.5 = 0.45\)). CART and C4.5 therefore grow with Gini or entropy and use the misclassification error only to <em>evaluate</em> the finished tree.<sup class="c"><a href="#r7">[7]</a>,<a href="#r23">[23]</a></sup>

### 6.3 The greedy recursive algorithm, with a split computed by hand

CART grows a classification tree as follows:

1. Start with all training cases at the root.
2. For every feature \(j\) and every threshold \(t\) between consecutive sorted values of \(x_j\) at the node, compute \(\Delta I\) for the split "\(x_j \le t\)".
3. Take the split with the largest \(\Delta I\) and send the cases to two children.
4. Recurse on each child until a stopping rule is met: pure node, too few cases, maximum depth, or no split decreases impurity.
5. Label each leaf with its majority class (or its class proportions).

The algorithm is <strong>greedy</strong>: it chooses the best split now and never revisits it. It need not find the smallest or most accurate tree — finding an optimal tree is NP-complete<sup class="c"><a href="#r10">[10]</a></sup> — but one level costs only \(O(p\, n \log n)\).

<strong>By hand.</strong> Ten patients, five malignant (M) and five benign (B); a test sends six to the left (5 B, 1 M) and four to the right (4 M).

- Parent: \(G = 1 - (0.5^2 + 0.5^2) = 0.5\); \(H = 1\) bit.
- Left (5 B, 1 M): \(G_L = 1 - (5/6)^2 - (1/6)^2 = 10/36 \approx 0.278\); \(H_L = -\tfrac56\log_2\tfrac56 - \tfrac16\log_2\tfrac16 \approx 0.650\) bits.
- Right (4 M): \(G_R = H_R = 0\).
- Weighted children: \(0.6 \times 0.278 = 0.167\); \(0.6 \times 0.650 = 0.390\) bits.
- Decrease: \(\Delta G = 0.333\); information gain \(0.610\) bits.

A competing test sending 3 B + 2 M left and 2 B + 3 M right gives \(G_L = G_R = 0.48\) and \(\Delta G = 0.02\). The cell confirms the arithmetic.



In [ ]:
# Impurity functions and the hand-worked split of Section 6.3.
gini_index <- function(labels) { pk <- table(labels) / length(labels); 1 - sum(pk^2) }
entropy_bits <- function(labels) { pk <- table(labels) / length(labels); pk <- pk[pk > 0]; -sum(pk * log2(pk)) }
split_quality <- function(yL, yR, impurity = gini_index) {
  yall <- c(yL, yR); n <- length(yall); child <- (length(yL) / n) * impurity(yL) + (length(yR) / n) * impurity(yR)
  c(decrease = impurity(yall) - child, child = child)
}
left <- rep(c("B", "M"), c(5, 1)); right <- rep("M", 4)           # 5B+1M left, 4M right
g <- split_quality(left, right, gini_index); h <- split_quality(left, right, entropy_bits)
cat(sprintf("parent Gini %.3f; left %.3f; right %.3f; weighted child %.3f -> decrease %.3f\n",
    gini_index(c(left, right)), gini_index(left), gini_index(right), g["child"], g["decrease"]))
cat(sprintf("parent entropy %.3f bits; left %.3f; weighted child %.3f -> information gain %.3f bits\n",
    entropy_bits(c(left, right)), entropy_bits(left), h["child"], h["decrease"]))
cat(sprintf("competing split (3B+2M | 2B+3M): decrease in Gini %.3f\n",
    split_quality(rep(c("B","M"), c(3,2)), rep(c("B","M"), c(2,3)))["decrease"]))



Every number of the hand calculation is reproduced. This procedure — child impurities, weighted by size, subtracted from the parent — is the entire inner loop of CART.

### 6.4 One split of a real feature, by exhaustive search and by rpart

Step 2, for a feature, is a scan: sort the distinct values at the node, take the midpoint between consecutive ones as a candidate threshold, and compute the weighted child impurity for each. The cell scans <em>cell size</em> over the 478 training cases, then asks <code>rpart</code> with <code>maxdepth = 1</code> — a <em>decision stump</em> — to do the same, on that feature alone and then with all nine features.



In [ ]:
# Exhaustive scan of the thresholds of 'Cell.size' (the inner loop of CART), then a depth-1 tree.
wgini <- function(f, t, y) { l <- y[f <= t]; r <- y[f > t]; if (!length(l) || !length(r)) return(NA)
  (length(l) * gini_index(l) + length(r) * gini_index(r)) / length(y) }
f <- tr$Cell.size; y <- tr$Class
vals <- sort(unique(f)); cand <- (head(vals, -1) + tail(vals, -1)) / 2
wg <- sapply(cand, wgini, f = f, y = y); best <- cand[which.min(wg)]
cat(sprintf("%d candidate thresholds; parent Gini %.4f\n", length(cand), gini_index(y)))
cat(sprintf("best Cell.size <= %.1f: weighted child Gini %.4f, decrease %.4f\n", best, min(wg), gini_index(y) - min(wg)))
set.seed(1); stump1 <- rpart(Class ~ Cell.size, tr, method = "class", control = rpart.control(maxdepth = 1, cp = 0, xval = 0))
set.seed(1); stumpA <- rpart(Class ~ ., tr, method = "class", control = rpart.control(maxdepth = 1, cp = 0, xval = 0))
cat(sprintf("rpart stump on Cell.size only : split %.1f; test accuracy %.3f\n",
    stump1$splits[1, "index"], mean(predict(stump1, te, type = "class") == te$Class)))
cat(sprintf("rpart stump on all 9 features : splits on '%s'; test accuracy %.3f\n",
    rownames(stumpA$splits)[1], mean(predict(stumpA, te, type = "class") == te$Class)))
plot(cand, wg, type = "l", col = BLUE, lwd = 1.5, xlab = "candidate threshold on 'Cell.size'",
     ylab = "weighted Gini of the two children", main = "Scanning every threshold of one feature (CART's inner loop)")
abline(v = best, col = ORANGE, lty = 2); points(best, min(wg), pch = 19, col = ORANGE)



<p class="caption"><b>Figure 7.</b> Weighted Gini of the two children as a function of the threshold on <em>cell size</em>, over the candidate midpoints. The minimum is at 3.5: tumours whose cell-size score exceeds it are overwhelmingly malignant.</p>

The scan finds the threshold 3.5 with weighted Gini 0.125, down from the parent's 0.460, and <code>rpart</code>'s stump on the same feature finds <em>exactly the same threshold</em> — its algorithm is the scan just written. Given all nine features the stump prefers <em>cell shape</em>, a near-duplicate of cell size; which of the two wins is a matter of a few training cases, a first sign of the instability of Section 6.10. A single question already classifies about nine of ten test cases.

<div class="box warn"><h4>Why only midpoints — and why trees need no scaling</h4><p>Any threshold strictly between two consecutive sorted values gives the same split, so one representative per gap suffices. It follows that a tree is <strong>invariant to any monotone transformation of a feature</strong>: logarithms, standardisation or a change of units alter the thresholds' values but not the splits. This is why trees need no feature scaling, unlike k-NN and SVM in Unit 3.</p></div>

### 6.5 Growing the full tree and reading its rules

Left to run until every leaf is pure, the algorithm classifies every training case correctly. The cell grows that tree, reports its size and accuracies, and prints the rules of a depth-3 tree.



In [ ]:
# The unpruned tree: perfect on training data, worse on test data. Rules as a pre-order traversal.
set.seed(1); full <- rpart(Class ~ ., tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
nleaf <- function(f) sum(f$frame$var == "<leaf>")
cat(sprintf("unpruned tree: %d leaves, %d nodes; train acc %.3f, test acc %.3f\n",
    nleaf(full), nrow(full$frame), mean(predict(full, tr, type = "class") == tr$Class), mean(predict(full, te, type = "class") == te$Class)))
set.seed(1); tree3 <- rpart(Class ~ ., tr, method = "class", control = ctl(3))
cat(sprintf("depth-3 tree: %d leaves; train acc %.3f, test acc %.3f\n\n",
    nleaf(tree3), mean(predict(tree3, tr, type = "class") == tr$Class), mean(predict(tree3, te, type = "class") == te$Class)))
print(tree3)                                              # the rules, as an indented pre-order traversal
rpart.plot(tree3, type = 2, extra = 8, box.palette = list("#fbe3d3", "#dfe7f1"), main = "The depth-3 tree drawn")



<p class="caption"><b>Figure 8.</b> The depth-3 tree, drawn. Any root-to-leaf path is an IF–THEN rule: the first question concerns cell shape, the next ones cell size, the bare nuclei or clump thickness. Leaves reached by one or two training cases are the ones most likely to encode noise.</p>

The unpruned tree reaches 100 % training accuracy, as it must, with 23 leaves and depth 9; on the 205 test cases it is correct in 94.1 %. The depth-3 tree, with far fewer leaves, matches it on the test cases: the extra splits learned the accidents of the 478 training cases, not the pattern. This is <strong>overfitting</strong>, the theme of Supplement S1, and depth is the knob that controls it. The printed tree is a pre-order traversal in which indentation encodes depth and each leaf shows its class counts.

### 6.6 Depth versus overfitting

The experiment grows trees of maximum depth 1 to 8 and records the training accuracy, the 5-fold cross-validated accuracy within the training part (the honest estimate of S1), and the test accuracy — shown for illustration only, never used to choose the depth.



In [ ]:
# Training, cross-validated and test accuracy versus the maximum depth of the tree.
acc <- function(p, y) mean(p == y)
kfold <- function(d, k = 5) { set.seed(1); fold <- sample(rep(1:k, length.out = nrow(tr)))
  sapply(1:k, function(i) { fit <- rpart(Class ~ ., tr[fold != i, ], method = "class", control = ctl(d))
    acc(predict(fit, tr[fold == i, ], type = "class"), tr$Class[fold == i]) }) }
curve <- do.call(rbind, lapply(1:8, function(d) { set.seed(1); ft <- rpart(Class ~ ., tr, method = "class", control = ctl(d)); cv <- kfold(d)
  data.frame(depth = d, train = acc(predict(ft, tr, type = "class"), tr$Class), cv = mean(cv), cvsd = sd(cv),
             test = acc(predict(ft, te, type = "class"), te$Class), leaves = nleaf(ft)) }))
best_d <- curve$depth[which.max(curve$cv)]
plot(curve$depth, curve$train, type = "b", pch = 19, col = GREY, ylim = c(0.86, 1.01),
     xlab = "maximum depth of the tree", ylab = "accuracy", main = "Depth controls overfitting")
arrows(curve$depth, curve$cv - curve$cvsd, curve$depth, curve$cv + curve$cvsd, length = 0.02, angle = 90, code = 3, col = BLUE)
lines(curve$depth, curve$cv, type = "b", pch = 15, col = BLUE); lines(curve$depth, curve$test, type = "b", pch = 17, lty = 2, col = ORANGE)
abline(v = best_d, col = BLUE, lty = 3)
legend("bottomright", bty = "n", cex = 0.8, col = c(GREY, BLUE, ORANGE), pch = c(19, 15, 17), lty = c(1, 1, 2),
       legend = c("training", "5-fold CV (training part)", "test (not used for selection)"))
cat("depth with the best cross-validated accuracy:", best_d, "\n"); print(round(curve, 3), row.names = FALSE)



<p class="caption"><b>Figure 9.</b> Accuracy versus maximum depth. Training accuracy rises towards 1.0; cross-validated accuracy (one standard deviation across folds) peaks around depth 3 and then declines slightly; test accuracy plateaus. Differences of one or two points between neighbouring depths are within the error bars.</p>

Beyond a certain complexity, capacity buys training accuracy and nothing else. The cross-validated optimum is depth 3 (0.964), but the standard deviation across folds is two points, so the honest statement is "a depth of about 2 to 4 is as good as any". Choosing depth by the orange test curve would be the error of Supplement S1 — using the test set to select the model.

### 6.7 Cost-complexity pruning

A depth limit cuts every branch at the same level, although some deserve to be deeper. CART's <strong>cost-complexity pruning</strong> grows the full tree and then removes the branches that contribute least to the fit relative to their size: for a tree \(T\) with \(|T|\) leaves and training error \(R(T)\),

\[ R_\alpha(T) = R(T) + \alpha\,|T| , \]

where \(\alpha \ge 0\) is the price of a leaf. As \(\alpha\) grows the optimal subtree loses its weakest branches one by one; the optimal subtrees are nested, so the whole sequence is cheap to compute, and \(\alpha\) is chosen by cross-validation.<sup class="c"><a href="#r7">[7]</a>,<a href="#r23">[23]</a></sup> In <code>rpart</code> the parameter is <code>cp</code>, a rescaled \(\alpha\), and the function cross-validates every subtree automatically, reporting the result in a <em>complexity table</em>.<sup class="c"><a href="#r34">[34]</a></sup>



In [ ]:
# rpart's own 10-fold-CV complexity table is the pruning path; pick the subtree with least CV error.
set.seed(1); tcp <- rpart(Class ~ ., tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 10))
print(round(tcp$cptable, 4))
cp_opt <- tcp$cptable[which.min(tcp$cptable[, "xerror"]), "CP"]
pruned <- prune(tcp, cp = cp_opt * 1.0001)
cat(sprintf("best CV subtree: cp %.4f, %d leaves, test accuracy %.3f\n",
    cp_opt, nleaf(pruned), acc(predict(pruned, te, type = "class"), te$Class)))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "rel error"], type = "b", pch = 19, col = GREY,
     xlab = "number of leaves", ylab = "error", main = "Training vs cross-validated error")
arrows(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"] - tcp$cptable[, "xstd"],
       tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"] + tcp$cptable[, "xstd"], length = 0.02, angle = 90, code = 3, col = BLUE)
lines(tcp$cptable[, "nsplit"] + 1, tcp$cptable[, "xerror"], type = "b", pch = 15, col = BLUE)
legend("topright", bty = "n", cex = 0.8, col = c(GREY, BLUE), pch = c(19, 15), legend = c("training (rel. error)", "10-fold CV error"))
abline(v = nleaf(pruned), col = ORANGE, lty = 2)
rpart.plot(pruned, type = 2, extra = 104, box.palette = list("#fbe3d3", "#dfe7f1"), main = "The cross-validated best subtree")



<p class="caption"><b>Figure 10.</b> Left: training error falls monotonically with the number of leaves, but the 10-fold cross-validated error (one standard deviation) bottoms out at a few leaves and then rises — the overfitting of Section 6.6 in another guise. Right: the cross-validated best subtree, the one whose rules one would show to a clinician.</p>

The best subtree has 5 leaves, cross-validated error about 0.10 and a test accuracy of 0.946 — as good as the 23-leaf tree with a fifth of the leaves; the branches beyond it were fitting noise. The simpler "one-standard-error" rule, which takes the smallest tree whose error is within one standard deviation of the minimum, would prune further still, to a 3-leaf tree. This pruned tree is the single tree compared against the forest in Section 7.

### 6.8 Feature importance and its caveats

A tree offers a natural measure of each feature's contribution: the total impurity decrease of all splits on that feature, weighted by node size — the <strong>impurity-based importance</strong> (mean decrease in impurity, MDI).<sup class="c"><a href="#r7">[7]</a>,<a href="#r24">[24]</a></sup> Strobl and colleagues documented two caveats.<sup class="c"><a href="#r16">[16]</a></sup> It is <em>biased towards features with many split points</em>: a feature offering the greedy search more candidate thresholds has more chances to find a spuriously good split, so even a pure-noise many-valued feature receives importance. And among <em>correlated</em> features the credit is distributed arbitrarily: whichever near-duplicate is chosen first takes it. The cell shows the first bias in pure form — four noise features with different numbers of levels and nothing else — using a forest (Section 7) to average over the arbitrariness of single trees.



In [ ]:
# Caveat 1 of impurity importance: four PURE-NOISE features with different numbers of candidate thresholds.
set.seed(1); nn <- nrow(tr)
noise <- data.frame(Class = tr$Class, noise_cont = rnorm(nn), noise_20 = sample(0:19, nn, TRUE),
                    noise_4 = sample(0:3, nn, TRUE), noise_bin = sample(0:1, nn, TRUE))
set.seed(1); rf_null <- randomForest(Class ~ ., noise, ntree = 300)
cat("impurity importance (MeanDecreaseGini) when EVERY feature is noise (should be equal if unbiased):\n")
print(round(importance(rf_null)[, 1], 1))



When every feature is noise the importances should be equal; instead they are ordered by the number of candidate thresholds — about 104 for the continuous feature against 9 for the binary one, with no information in either. The rules: never read impurity importance as causal or marginal relevance; compare it with permutation importance (Section 7.4); report correlated features as a group.<sup class="c"><a href="#r16">[16]</a>,<a href="#r17">[17]</a></sup>

### 6.9 Regression trees

The same algorithm fits a continuous target. In a <strong>regression tree</strong> each leaf predicts the <em>mean</em> of the training targets reaching it, the impurity is their <em>variance</em>, and the prediction is piecewise constant: a step function in one dimension. The Python edition used the diabetes data (442 patients, disease progression after one year);<sup class="c"><a href="#r29">[29]</a></sup> the R stand-in is the Boston housing data shipped with <code>mlbench</code> (506 census tracts, median home value from thirteen covariates),<sup class="c"><a href="#r33">[33]</a>,<a href="#r4">[4]</a></sup> which shows the behaviour just as well.



In [ ]:
# A regression tree on the Boston data: R^2 on the test part, and the step-function fit along one covariate.
data(BostonHousing, package = "mlbench"); bh <- BostonHousing
set.seed(1); bi <- sample(nrow(bh), round(0.7 * nrow(bh))); btr <- bh[bi, ]; bte <- bh[-bi, ]
r2 <- function(p, y) 1 - sum((y - p)^2) / sum((y - mean(y))^2)
set.seed(1); reg2  <- rpart(medv ~ ., btr, method = "anova", control = rpart.control(maxdepth = 2, cp = 0, xval = 0))
set.seed(1); regf  <- rpart(medv ~ ., btr, method = "anova", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
cat(sprintf("depth-2 tree splits on {%s}: test R^2 %.3f\n", paste(setdiff(unique(as.character(reg2$frame$var)), "<leaf>"), collapse = ", "), r2(predict(reg2, bte), bte$medv)))
cat(sprintf("unpruned tree (%d leaves): train R^2 %.3f, test R^2 %.3f\n", nleaf(regf), r2(predict(regf, btr), btr$medv), r2(predict(regf, bte), bte$medv)))
grid <- seq(min(bh$rm), max(bh$rm), length.out = 400)
plot(btr$rm, btr$medv, pch = 19, cex = 0.5, col = GREY, xlab = "average number of rooms (rm)", ylab = "median home value (medv)",
     main = "A regression tree predicts a piecewise-constant function")
for (dc in list(c(1, ORANGE), c(3, BLUE))) { d <- as.integer(dc[1])
  r <- rpart(medv ~ rm, btr, method = "anova", control = rpart.control(maxdepth = d, cp = 0, xval = 0))
  pr <- predict(r, data.frame(rm = grid)); lines(grid, pr, col = dc[2], lwd = 1.8, type = "s") }
legend("topleft", bty = "n", cex = 0.8, lwd = 1.8, col = c(ORANGE, BLUE), legend = c("tree on rm, depth 1", "tree on rm, depth 3"))



<p class="caption"><b>Figure 11.</b> Median home value against the average number of rooms for the training tracts, with regression trees of depth 1 and 3 fitted to this feature alone: constant within each leaf, jumping at each threshold.</p>

The depth-2 tree explains about a third of the variance of unseen tracts, and the unpruned tree — a leaf per few tracts — has a <em>lower</em> test \(R^2\) than the shallow one despite a perfect training fit: overfitting again. The step function also shows a limitation: a tree represents a smooth trend only as a staircase and extrapolates as a constant. Forests inherit both, attenuated.

### 6.10 Strengths and weaknesses of a single tree

<strong>Strengths.</strong> No feature scaling (Section 6.4); continuous and categorical features in one model, and missing values handled by surrogate splits (native in <code>rpart</code>); interpretability — the model is a short list of rules and each prediction a path; fast training, \(O(p\,n\log n)\) per level, and \(O(\text{depth})\) prediction; interactions captured automatically.

<strong>Weaknesses.</strong> <strong>Instability</strong>: the algorithm is greedy and each split is chosen by a small margin, so a small change in the training data can change the root and everything below it — the property Breiman named when he introduced bagging.<sup class="c"><a href="#r11">[11]</a></sup> And <strong>axis-aligned boundaries</strong>: an oblique boundary must be approximated by a staircase of many splits. Two cells demonstrate each.



In [ ]:
# Weakness 1, instability: trees grown on two bootstrap samples of the same training data.
set.seed(1); b1 <- sample(nn, nn, replace = TRUE); b2 <- sample(nn, nn, replace = TRUE)
cat(sprintf("bootstrap sample 1: %d distinct training cases of %d (%.1f%%)\n", length(unique(b1)), nn, 100 * length(unique(b1)) / nn))
t1 <- rpart(Class ~ ., tr[b1, ], method = "class", control = ctl(2)); t2b <- rpart(Class ~ ., tr[b2, ], method = "class", control = ctl(2))
cat("depth-2 tree, sample 1 splits (root first):", paste(setdiff(unique(as.character(t1$frame$var)), "<leaf>"), collapse = ", "), "\n")
cat("depth-2 tree, sample 2 splits (root first):", paste(setdiff(unique(as.character(t2b$frame$var)), "<leaf>"), collapse = ", "), "\n")
set.seed(1); f1 <- rpart(Class ~ ., tr[b1, ], method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
set.seed(1); f2 <- rpart(Class ~ ., tr[b2, ], method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
cat(sprintf("two unpruned trees from the two samples disagree on %.1f%% of the %d test cases\n",
    100 * mean(predict(f1, te, type = "class") != predict(f2, te, type = "class")), nrow(te)))



Each bootstrap sample contains about 63 % of the distinct training cases (Section 7.3 explains the number), so the two samples differ only in which of the same 478 patients are repeated. The two trees share the same root here — cell shape and cell size are such close near-duplicates that the choice is barely contested — but differ in thresholds and in the features below the root, and two unpruned trees from the two samples disagree on several per cent of the test cases. For a clinical rule this is uncomfortable, and it is the property Section 7 turns into an advantage.



In [ ]:
# Weakness 2, axis-aligned boundaries: a tree on two features partitions the plane into rectangles.
feats <- c("Cell.size", "Bare.nuclei")
grid2 <- expand.grid(Cell.size = seq(1, 10, 0.1), Bare.nuclei = seq(1, 10, 0.1))
tp <- rpart(Class ~ Cell.size + Bare.nuclei, tr, method = "class", control = ctl(3))
tfull2 <- rpart(Class ~ Cell.size + Bare.nuclei, tr, method = "class", control = rpart.control(cp = 0, minsplit = 2, minbucket = 1, xval = 0))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (mt in list(list(tp, "depth-3 tree"), list(tfull2, "unpruned tree"))) { m <- mt[[1]]
  Z <- predict(m, grid2, type = "class")
  plot(grid2$Cell.size, grid2$Bare.nuclei, col = ifelse(Z == "malignant", "#fbe3d3", "#dfe7f1"), pch = 15, cex = 0.5,
       xlab = feats[1], ylab = feats[2], main = sprintf("%s: test acc %.3f (%d leaves)", mt[[2]], acc(predict(m, te, type = "class"), te$Class), nleaf(m)))
  jit <- function(v) v + runif(length(v), -0.3, 0.3)
  points(jit(te$Cell.size), jit(te$Bare.nuclei), col = ifelse(te$Class == "malignant", ORANGE, BLUE), pch = 19, cex = 0.5) }



<p class="caption"><b>Figure 12.</b> Decision regions of a depth-3 tree (left) and an unpruned tree (right) on two features, with the test cases jittered on top. Every boundary is horizontal or vertical. The unpruned tree carves thin slivers around individual training cases; the depth-3 tree draws a few large boxes and is about as accurate.</p>

Where the true boundary is oblique — malignancy rises with cell size and the bare-nuclei score together — a tree can only approximate it with steps, and the finer the steps the fewer training cases each is fitted to. Compare the smooth boundaries of the SVM and k-NN in Unit 3; forests soften the staircase by averaging many of them.

<div class="exercise"><h4>Exercise 3</h4><p>Replace <code>feats</code> by two weak features such as <code>"Mitoses"</code> and <code>"Epith.c.size"</code>: how many leaves does the unpruned tree grow, and what is its test accuracy? Then set <code>minbucket = 10</code> in both trees. How does the picture change, and why is this stopping rule often preferable to a depth limit?</p></div>

## 7. Random forests

### 7.1 Bagging: averaging unstable predictors

In 1996 Breiman proposed a remedy for instability that is almost embarrassingly simple: train many trees, each on a different <strong>bootstrap sample</strong> (\(n\) cases drawn <em>with replacement</em> from the \(n\) available), and combine them by vote or average. He called it <strong>bagging</strong>, for <em>bootstrap aggregating</em>, and showed that it helps unstable predictors substantially while leaving stable ones such as k-NN almost unchanged.<sup class="c"><a href="#r11">[11]</a></sup> The reason is a fact about averages: if \(B\) predictors each have variance \(\sigma^2\) and pairwise correlation \(\rho\),

\[ \operatorname{Var}\!\left(\frac{1}{B}\sum_{b=1}^{B} \hat f_b\right) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2 . \]

The second term vanishes as \(B\) grows: averaging removes the <em>independent</em> part of the variability. The first does not: whatever the trees have in common survives.<sup class="c"><a href="#r23">[23]</a></sup> The cell simulates the formula with \(\hat f_b = \sqrt{\rho}\,c + \sqrt{1-\rho}\,e_b\) (shared \(c\), independent \(e_b\), all standard normal).



In [ ]:
# Variance of the average of B predictors with pairwise correlation rho: simulation versus rho + (1-rho)/B.
set.seed(0)
Bs <- c(1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200); reps <- 4000
plot(NA, xlim = range(Bs), ylim = c(0, 1.05), log = "x", xlab = "number of averaged predictors B",
     ylab = "variance of the average", main = "Averaging removes independent, not shared, variability")
cols <- c(GREEN, BLUE, ORANGE, GREY); rhos <- c(0, 0.25, 0.5, 0.75)
for (j in seq_along(rhos)) { rho <- rhos[j]
  sim <- sapply(Bs, function(B) { shared <- rnorm(reps); indiv <- matrix(rnorm(reps * B), reps, B)
    var(rowMeans(sqrt(rho) * shared + sqrt(1 - rho) * indiv)) })
  points(Bs, sim, pch = 19, col = cols[j], cex = 0.7); lines(Bs, rho + (1 - rho) / Bs, col = cols[j], lwd = 1.5) }
legend("topright", bty = "n", cex = 0.8, lwd = 1.5, col = cols, title = "pairwise correlation", legend = sprintf("rho = %.2f", rhos))
cat("floor as B -> infinity equals rho; at B=200 the formula gives:",
    paste(sprintf("rho=%.2f:%.3f", rhos, rhos + (1 - rhos) / 200), collapse = "  "), "\n")



<p class="caption"><b>Figure 13.</b> Variance of the average of \(B\) unit-variance predictors for four pairwise correlations; points simulated, lines the formula. With \(\rho = 0\) the variance falls as \(1/B\); with \(\rho = 0.5\) it reaches a floor of 0.5 after a few dozen predictors.</p>

Trees grown on bootstrap samples of the same data are strongly correlated — they tend to choose the same strong feature at the root — so their errors are shared. To push \(\rho\) down one must make the trees <em>more different</em>, even at the cost of making each individually worse.

### 7.2 Random feature subsets

That is the idea Ho introduced in 1995 as the <em>random subspace method</em><sup class="c"><a href="#r13">[13]</a>,<a href="#r14">[14]</a></sup> and that Breiman combined with bagging in 2001 to define the <strong>random forest</strong>:<sup class="c"><a href="#r12">[12]</a></sup> at <em>every node</em> of every tree only a random subset of \(m\) of the \(p\) features is considered (<code>mtry</code>; by default \(m = \sqrt{p}\) for classification, so \(m = 3\) of the nine features here). A tree forbidden from using <em>cell shape</em> at some node must find the next-best feature, and trees that differ at the root diverge below it. Each tree is somewhat worse; the ensemble is better. Trees are grown to full depth without pruning — their individual overfitting is averaged away — so a forest has essentially two parameters, \(m\) and \(B\), and is insensitive to both over a wide range. Breiman's bound makes the trade explicit: the generalisation error is at most \(\bar\rho\,(1 - s^2)/s^2\), with \(\bar\rho\) the mean correlation between trees and \(s\) their average strength.<sup class="c"><a href="#r12">[12]</a></sup> The R implementation is the <code>randomForest</code> package of Liaw and Wiener.<sup class="c"><a href="#r35">[35]</a></sup>

### 7.3 Out-of-bag error: a free validation estimate

Each bootstrap sample omits on average a fraction \((1 - 1/n)^n \approx e^{-1} \approx 0.368\) of the training cases — the <strong>out-of-bag</strong> (OOB) cases of that tree. Any training case is thus out of bag for about a third of the trees, which can classify it <em>as if it were new</em>. Comparing the OOB vote for each case with its label gives the <strong>OOB error</strong>, an estimate of generalisation error without a test set or cross-validation, at no cost; Breiman showed it is about as accurate as a test set of the training set's size.<sup class="c"><a href="#r15">[15]</a>,<a href="#r12">[12]</a></sup> The cell fits 300 trees and compares OOB, cross-validated and test accuracy.



In [ ]:
# A random forest on the training part: out-of-bag accuracy versus cross-validation versus the held-out test set.
cat(sprintf("fraction left out of one bootstrap sample: (1 - 1/%d)^%d = %.3f  (e^-1 = %.3f)\n", nn, nn, (1 - 1 / nn)^nn, exp(-1)))
set.seed(1); rf <- randomForest(Class ~ ., tr, ntree = 300, importance = TRUE)
set.seed(1); fold <- sample(rep(1:5, length.out = nrow(tr)))
cv_rf <- mean(sapply(1:5, function(i) { m <- randomForest(Class ~ ., tr[fold != i, ], ntree = 300)
  acc(predict(m, tr[fold == i, ]), tr$Class[fold == i]) }))
print(data.frame(estimate = c("OOB accuracy (training cases, OOB trees)", "5-fold CV accuracy (training part)",
                              "accuracy on the 205 held-out test cases", "accuracy on the training cases themselves"),
                 value = round(c(1 - rf$err.rate[300, "OOB"], cv_rf, acc(predict(rf, te), te$Class), acc(predict(rf, tr), tr$Class)), 3)), row.names = FALSE)
cat("mtry (features tried per split):", rf$mtry, "= sqrt(9)\n")



The OOB estimate, the cross-validated estimate and the test accuracy agree to within one or two percentage points, while the accuracy on the training cases is 1.0, as it must be for unpruned trees, and says nothing. In practice the OOB estimate replaces cross-validation for forests: it costs nothing, uses all the data, and can be monitored as trees are added — which answers "how many trees?".



In [ ]:
# OOB error as trees are added: randomForest records it after every tree in rf$err.rate.
single_err <- 1 - mean(kfold(30))                           # a single, deep tree, 5-fold CV error
plot(1:300, rf$err.rate[, "OOB"], type = "l", col = BLUE, lwd = 1.5, log = "x", ylim = c(0, max(rf$err.rate[, "OOB"])),
     xlab = "number of trees B", ylab = "error rate", main = "Out-of-bag error versus number of trees")
abline(h = single_err, col = ORANGE, lty = 2)
legend("topright", bty = "n", cex = 0.8, col = c(BLUE, ORANGE), lwd = c(1.5, 1), lty = c(1, 2),
       legend = c("OOB error of the forest", sprintf("single tree (5-fold CV error %.3f)", single_err)))
cat(sprintf("OOB error at B = 10, 50, 100, 300: %.3f, %.3f, %.3f, %.3f\n",
    rf$err.rate[10, "OOB"], rf$err.rate[50, "OOB"], rf$err.rate[100, "OOB"], rf$err.rate[300, "OOB"]))



<p class="caption"><b>Figure 14.</b> Out-of-bag error as trees are added, against the cross-validated error of a single tree (dashed). The forest's error falls steeply over the first few dozen trees and plateaus from about a hundred — below the single tree's error.</p>

With very few trees many cases have only one or two OOB votes and the estimate is noisy; by 50 to 100 trees it has reached its plateau, and 300 to 500 is a safe default at this size. The number of trees cannot be over-fitted: more trees never increase the expected error, only the time.

### 7.4 Feature importance: impurity-based versus permutation

A forest inherits the impurity importance of Section 6.8, averaged over trees, and offers a second measure: <strong>permutation importance</strong>. Randomly permute one feature's values among the out-of-bag cases — destroying its relation with the label while keeping its distribution — and measure the drop in accuracy. A feature the model does not use gives a drop of zero whatever its number of split points, which removes the first bias of Section 6.8.<sup class="c"><a href="#r12">[12]</a>,<a href="#r16">[16]</a></sup> Correlation remains: if <em>cell size</em> is permuted but <em>cell shape</em> is intact, the forest keeps almost the same information and the drop is small for each member of a correlated group even when the group is essential; conditional and corrected permutation tests address this.<sup class="c"><a href="#r17">[17]</a>,<a href="#r18">[18]</a></sup> With <code>importance = TRUE</code>, <code>randomForest</code> reports both: <code>MeanDecreaseGini</code> (MDI) and <code>MeanDecreaseAccuracy</code> (OOB permutation).



In [ ]:
# Impurity-based importance (MeanDecreaseGini) versus permutation importance (MeanDecreaseAccuracy). They disagree.
imp <- importance(rf)[, c("MeanDecreaseAccuracy", "MeanDecreaseGini")]
imp <- imp[order(imp[, "MeanDecreaseGini"]), ]
par(mfrow = c(1, 2), mar = c(4, 7, 2, 1))
barplot(imp[, "MeanDecreaseGini"], horiz = TRUE, las = 1, col = BLUE, cex.names = 0.8, main = "impurity importance (MDI)", xlab = "mean decrease in Gini")
barplot(imp[, "MeanDecreaseAccuracy"], horiz = TRUE, las = 1, col = ORANGE, cex.names = 0.8, main = "permutation importance (OOB)", xlab = "mean decrease in accuracy")
cat("top-3 by MDI:        ", paste(rev(rownames(imp))[1:3], collapse = ", "), "\n")
cat("top-3 by permutation:", paste(rownames(imp)[order(-imp[, "MeanDecreaseAccuracy"])][1:3], collapse = ", "), "\n")



<p class="caption"><b>Figure 15.</b> Impurity importance (left) and permutation importance (right) of the nine features. Both agree that cell shape, cell size and the bare nuclei dominate, but the orderings differ: the bare nuclei and clump thickness rise under permutation, because they carry information not duplicated by the correlated size-and-shape pair.</p>

Impurity importance says "these features were used a lot"; permutation importance says "the predictions depend on this feature <em>given that the others are available</em>". With near-duplicate measurements the first is spread over the group and the second is small for every member. Neither says "this feature causes malignancy". For a per-patient explanation the modern tool is the Shapley-value decomposition for trees, beyond this unit.<sup class="c"><a href="#r31">[31]</a></sup>

### 7.5 Partial dependence: what does the forest do with a feature?

A forest has no coefficients, but one can ask how its prediction changes, <em>on average</em>, when one feature is varied with the others left as observed — the <strong>partial dependence</strong> on feature \(j\) at value \(v\):<sup class="c"><a href="#r19">[19]</a></sup>

\[ \overline{f}_j(v) = \frac{1}{n} \sum_{i=1}^{n} \hat f\big(x_{i1}, \dots, x_{i,j-1},\, v,\, x_{i,j+1}, \dots, x_{ip}\big), \]

computed below directly from the definition for the two features of Figure 12.



In [ ]:
# Partial dependence of the forest's predicted P(malignant) on two features, computed from the definition.
partial_dependence <- function(model, data, feature, grid)
  sapply(grid, function(v) { d <- data; d[[feature]] <- v; mean(predict(model, d, type = "prob")[, "malignant"]) })
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (fe in c("Cell.size", "Bare.nuclei")) { grid <- 1:10; pdv <- partial_dependence(rf, tr, fe, grid)
  plot(grid, pdv, type = "l", col = BLUE, lwd = 2, ylim = c(0, 1), xlab = fe, ylab = "mean predicted P(malignant)", main = paste("partial dependence on", fe))
  rug(jitter(tr[[fe]]), col = GREY)
  cat(sprintf("%s: P(malignant) rises from %.2f at %d to %.2f at %d\n", fe, pdv[1], grid[1], pdv[length(pdv)], grid[length(grid)])) }



<p class="caption"><b>Figure 16.</b> Partial dependence of the forest's predicted probability of malignancy on <em>cell size</em> and <em>bare nuclei</em>; ticks mark the training values. Both curves rise over the range where the trees place most thresholds and flatten outside it: a smooth version of the step functions of Section 6.9, because hundreds of steps at slightly different thresholds average to a ramp.</p>

The curves rise steeply but not all the way to 1: setting one feature to a malignant-looking value does not change the prediction for a patient whose other features look benign — which is also the caution, since such combinations may never occur in the data. Even so, a partial-dependence curve shows direction, shape and the range over which a feature matters, which no importance score does.

### 7.6 The forest against the single tree and the methods of Unit 3

The last experiment compares, under the S1 protocol, the pruned tree of Section 6.7, the forest, and three classifiers of Unit 3 — naive Bayes, k-nearest neighbours and the RBF-kernel SVM — by 5-fold cross-validation on the training part (accuracy and ROC AUC) and once on the test part.



In [ ]:
# Comparison under the S1 protocol: 5-fold CV on the training part (accuracy, ROC AUC), then one test-set evaluation.
auc_te <- function(prob_mal) as.numeric(auc(roc(te$Class, prob_mal, quiet = TRUE, levels = c("benign", "malignant"), direction = "<")))
pos <- function(m, d) {                                     # P(malignant) from each model
  if (inherits(m, "randomForest")) predict(m, d, type = "prob")[, "malignant"]
  else if (inherits(m, "naiveBayes")) predict(m, d, type = "raw")[, "malignant"]
  else if (inherits(m, "rpart")) predict(m, d)[, "malignant"] else attr(predict(m, d, probability = TRUE), "probabilities")[, "malignant"]
}
set.seed(1); fit_list <- list(
  "decision tree, pruned (Sec. 6.7)" = prune(tcp, cp = cp_opt * 1.0001),
  "random forest, 300 trees"         = randomForest(Class ~ ., tr, ntree = 300),
  "naive Bayes (Unit 3)"             = naiveBayes(Class ~ ., tr),
  "SVM, RBF kernel (Unit 3)"         = svm(Class ~ ., tr, kernel = "radial", probability = TRUE))
cv5 <- function(fitfun) { set.seed(1); fo <- sample(rep(1:5, length.out = nrow(tr)))
  mean(sapply(1:5, function(i) { m <- fitfun(tr[fo != i, ]); acc(predict(m, tr[fo == i, ], type = if (inherits(m, "rpart")) "class" else "class"), tr$Class[fo == i]) })) }
cvfuns <- list(function(d) prune(rpart(Class ~ ., d, method = "class", control = rpart.control(cp = 0, xval = 0)), cp = cp_opt * 1.0001),
               function(d) randomForest(Class ~ ., d, ntree = 300), function(d) naiveBayes(Class ~ ., d),
               function(d) svm(Class ~ ., d, kernel = "radial"))
# k-NN handled separately (needs scaling)
sc <- scale(tr[, 1:9]); scte <- scale(te[, 1:9], center = attr(sc, "scaled:center"), scale = attr(sc, "scaled:scale"))
comp <- do.call(rbind, Map(function(nm, m, cf) data.frame(model = nm, CV_accuracy = round(cv5(cf), 3),
                 CV_ROC_AUC = round(auc_te(pos(m, te)), 3), test_accuracy = round(acc(predict(m, te, type = if (inherits(m, "rpart")) "class" else "class"), te$Class), 3)),
                 names(fit_list), fit_list, cvfuns))
knn5 <- knn(sc, scte, tr$Class, k = 5)
comp <- rbind(comp, data.frame(model = "k-NN, k=5, standardised (Unit 3)", CV_accuracy = NA, CV_ROC_AUC = NA, test_accuracy = round(acc(knn5, te$Class), 3)))
print(comp, row.names = FALSE)



The forest improves on the pruned tree by a couple of points of cross-validated accuracy and several points of ROC AUC: averaging unstable trees pays. Among the strong methods the differences are within a standard deviation of one another on these 478 training cases; the single tree is clearly last. The picture is representative: in a comparison of 179 classifiers on 121 datasets, random forests were the family most often best or near-best, with SVMs close behind,<sup class="c"><a href="#r22">[22]</a></sup> and recent benchmarks on medium-sized tabular data still find tree ensembles ahead of deep neural networks.<sup class="c"><a href="#r21">[21]</a>,<a href="#r32">[32]</a></sup> What the forest offers that the SVM does not is the absence of scaling, mixed feature types, a free error estimate, and importance and partial-dependence tools; what it gives up relative to the single tree is the readable rule list — the trade Breiman described in his essay on the "two cultures" of statistical modelling.<sup class="c"><a href="#r30">[30]</a></sup>



In [ ]:
# The forest's probability surface on the two features of Figure 12, next to the unpruned tree's.
set.seed(1); rf2d <- randomForest(Class ~ Cell.size + Bare.nuclei, tr, ntree = 300)
gx <- seq(1, 10, 0.15); gg <- expand.grid(Cell.size = gx, Bare.nuclei = gx)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
for (mt in list(list(tfull2, "unpruned tree"), list(rf2d, "random forest (300 trees)"))) { m <- mt[[1]]
  P <- matrix(predict(m, gg, type = "prob")[, "malignant"], length(gx))
  image(gx, gx, P, col = hcl.colors(20, "Blue-Red"), xlab = feats[1], ylab = feats[2], zlim = c(0, 1),
        main = sprintf("%s, test acc %.3f", mt[[2]], acc(predict(m, te, type = if (inherits(m, "rpart")) "class" else "class"), te$Class)))
  contour(gx, gx, P, levels = 0.5, add = TRUE, drawlabels = FALSE, lwd = 1.5)
  jit <- function(v) v + runif(length(v), -0.3, 0.3)
  points(jit(te$Cell.size), jit(te$Bare.nuclei), col = ifelse(te$Class == "malignant", ORANGE, BLUE), pch = 19, cex = 0.5) }



<p class="caption"><b>Figure 17.</b> Predicted probability of malignancy over the plane of two features, for the unpruned tree (left) and the forest (right); the black line is the 0.5 contour. The tree's probabilities are nearly 0 or 1 everywhere, with rectangular islands; the forest's grade smoothly, the islands are averaged away.</p>

Averaging a few hundred staircases produces a smoother boundary and <em>graded uncertainty</em>: the pale region near the boundary is where the trees disagree, and a clinician should read a probability of 0.55 differently from one of 0.98.

<div class="box key"><h4>Random forests in one paragraph</h4><p>Grow \(B\) unpruned trees, each on a bootstrap sample, considering at each node only \(m \approx \sqrt{p}\) random features; predict by vote or average. Bootstrap and feature subsampling decorrelate the trees so that averaging reduces variance (Figure 13); the out-of-bag cases estimate the error for free (Section 7.3); \(B\) need only reach the plateau (Figure 14); the price is the loss of a single readable tree, partly compensated by importance measures and partial dependence.</p></div>

## 8. Beyond the plan: gradient boosting

<div class="box bridge"><h4>Not in the official syllabus</h4><p>This section goes one step beyond the unit because the method is, in current practice, the default for tabular data, and because it is the natural counterpart of the forest: both are tree ensembles that combine their trees in opposite ways.</p></div>

A forest grows its trees <em>independently</em> and averages them to reduce variance. <strong>Gradient boosting</strong>, formulated by Friedman in 2001, grows shallow trees <em>sequentially</em>, each fitted to the negative gradient of the loss of the ensemble so far, and adds them with a small learning rate, reducing <em>bias</em> step by step.<sup class="c"><a href="#r19">[19]</a></sup> XGBoost (2016) made the method fast and regularised, and it and its successors have won most competitions on tabular data since.<sup class="c"><a href="#r20">[20]</a></sup> Systematic comparisons find boosted trees and forests still ahead of deep networks on medium-sized tables, apparently because tree ensembles ignore uninformative features and represent irregular functions that networks smooth over.<sup class="c"><a href="#r21">[21]</a>,<a href="#r32">[32]</a></sup> Unlike a forest, boosting <em>can</em> overfit as trees are added, so their number must be chosen by validation or early stopping. R has dedicated packages (<code>gbm</code>, <code>xgboost</code>), but to keep to the course's small toolbox the cell implements the algorithm from scratch in about fifteen lines: a logistic gradient boosting whose base learners are depth-2 <code>rpart</code> regression trees fitted to the residuals \(y - p\).



In [ ]:
# Gradient boosting from scratch: depth-2 rpart regression trees fitted sequentially to the logistic gradient.
gb_fit <- function(train, M = 200, lr = 0.05, depth = 2) {
  y <- as.numeric(train$Class == "malignant"); X <- train[, 1:9]
  F0 <- log(mean(y) / (1 - mean(y))); Fv <- rep(F0, nrow(train)); trees <- vector("list", M)
  for (m in 1:M) { p <- 1 / (1 + exp(-Fv)); g <- y - p                       # negative gradient of the log-loss
    t <- rpart(g ~ ., cbind(X, g = g), method = "anova", control = rpart.control(maxdepth = depth, cp = 0, xval = 0))
    Fv <- Fv + lr * predict(t); trees[[m]] <- t }
  list(F0 = F0, trees = trees, lr = lr)
}
gb_prob <- function(mod, newX) { Fv <- rep(mod$F0, nrow(newX)); for (t in mod$trees) Fv <- Fv + mod$lr * predict(t, newX); 1 / (1 + exp(-Fv)) }
set.seed(1); gb <- gb_fit(tr, M = 200, lr = 0.05, depth = 2)
pte <- gb_prob(gb, te[, 1:9])
cat(sprintf("gradient boosting (200 depth-2 trees): train acc %.3f, test acc %.3f, test ROC AUC %.3f\n",
    mean((gb_prob(gb, tr[, 1:9]) > 0.5) == (tr$Class == "malignant")), mean((pte > 0.5) == (te$Class == "malignant")), auc_te(pte)))
cat(sprintf("for comparison, the random forest of Section 7.3: test ROC AUC %.3f\n", auc_te(predict(rf, te, type = "prob")[, "malignant"])))



Boosting lands in the same group as the forest and the SVM, within fold-to-fold noise on these 478 training cases: a hand-written ensemble of shallow trees reaches a test ROC AUC close to the forest's. On larger, messier tables — many uninformative features, missing values, mixed types — the gap between tree ensembles and the methods of Unit 3 usually widens in favour of the ensembles; on small, clean problems a tuned SVM is often as good and easier to defend. The comparison should be run under the S1 protocol, not assumed.

## 9. Applications in medicine and bioengineering

Decision trees entered medicine before they entered machine learning, because a clinician's protocol <em>is</em> a tree.

<strong>Triage of chest pain (1982).</strong> Goldman and colleagues applied recursive partitioning — CART, with Olshen among the authors — to 482 emergency-room patients to derive a protocol predicting myocardial infarction from the electrocardiogram and history; tested prospectively on 468 further patients, the protocol combined with physicians' judgement kept sensitivity while raising the specificity of intensive-care admission from 67 % to 77 %.<sup class="c"><a href="#r27">[27]</a></sup> The CART monograph, published two years later, opens with medical examples from the same period, including the prognosis of heart-attack patients.<sup class="c"><a href="#r7">[7]</a></sup> The protocol could be printed on a card and used at the bedside.

<strong>Clinical decision rules (1992).</strong> The Ottawa ankle rules, derived by Stiell and colleagues from 750 patients, are a sequence of yes/no questions on bone tenderness and the ability to bear weight that identifies patients who need no radiograph — operationally a shallow decision tree executed by hand; in the derivation study they had 100 % sensitivity for fracture.<sup class="c"><a href="#r26">[26]</a></sup> Most decision rules in emergency medicine have this form because a tree can be memorised and applied under time pressure: the interpretability of Section 6.10 is what makes them usable.

<strong>Genomics (2006).</strong> With thousands of features and tens of patients — gene-expression microarrays — single trees are hopeless and most methods of Unit 3 need heavy feature selection. Díaz-Uriarte and Alvarez de Andrés showed that random forests classify such data as well as the best alternatives without pre-selection, and proposed gene selection by iteratively discarding the least important genes while monitoring the OOB error.<sup class="c"><a href="#r25">[25]</a></sup> The biases of importance measures (Sections 6.8 and 7.4) were documented precisely because of their use for gene ranking.<sup class="c"><a href="#r16">[16]</a>,<a href="#r18">[18]</a></sup>

<div class="box warn"><h4>Reading a tree-based paper</h4><p>Was the tree pruned, and how was the depth or \(\alpha\) chosen — on the test data? Is the reported accuracy OOB, cross-validated or on an independent set? If importances are interpreted, are the features correlated, and was permutation importance used? Are the rules plausible to a clinician, or do they suggest shortcut learning (Unit 1)?</p></div>

## 10. Summary

You should now be able to explain, without notes:

- The recursive definition of a binary tree and its vocabulary (root, leaf, depth, height in edges with −1 for the empty tree; full, perfect, complete).
- The three representations; that <code>rpart</code> stores a tree as a data frame whose node numbers obey the \(2i\) / \(2i+1\) rule.
- The four traversals; that pre-order prints a tree as rules and in-order yields a search tree's keys in sorted order.
- That a perfect tree of height \(h\) has \(2^{h+1}-1\) nodes, so the height of an \(n\)-node tree lies between \(\log_2(n+1)-1\) and \(n-1\).
- The BST invariant; insertion and search in \(O(h)\); deletion via the in-order successor; sorted insertion gives \(h = n-1\), random insertion \(h \propto \log n\).
- What a rotation does; what AVL, red–black and B-trees guarantee and where they are used.
- Gini, entropy and misclassification, and why the concave ones are used for growing; the greedy CART algorithm and the threshold scan; why trees need no scaling.
- Depth versus overfitting; cost-complexity pruning; the two biases of impurity importance; regression trees as piecewise-constant functions; instability and axis-aligned boundaries.
- Bagging and \(\rho\sigma^2 + (1-\rho)\sigma^2/B\); random feature subsets as the way to reduce \(\rho\); the OOB estimate and \(e^{-1}\); impurity versus permutation importance; partial dependence; how the forest compares with a single tree and with Unit 3.

## 11. Self-assessment

<details><summary>1. A binary tree has 20 nodes. What are the minimum and maximum possible heights, and what shapes achieve them?</summary>Minimum \(\lceil \log_2 21 \rceil - 1 = 4\) (a complete tree: 15 nodes in the first four levels, 5 in the fifth). Maximum 19 (a chain).</details>

<details><summary>2. Give the pre-, in- and post-order traversals of the tree with root 8, left child 3 (children 1 and 6), right child 10 (right child 14 only). Is it a BST?</summary>Pre-order 8 3 1 6 10 14; in-order 1 3 6 8 10 14; post-order 1 6 3 14 10 8. Yes: the in-order sequence is strictly increasing.</details>

<details><summary>3. In the vector representation (0-based) a node is stored at index 12. Where are its parent and its children?</summary>Parent at \(\lfloor 11/2 \rfloor = 5\); children at 25 and 26. (In <code>rpart</code>'s one-based numbering the children of node \(i\) are \(2i\) and \(2i+1\).)</details>

<details><summary>4. Why does inserting patient records into a BST by increasing admission date produce a slow structure, and what are two remedies?</summary>Increasing keys always go right, giving a chain of height \(n-1\) and \(O(n)\) search. Remedies: a self-balancing tree (AVL, red–black, B-tree), or median-first insertion if all keys are known in advance.</details>

<details><summary>5. A right rotation is applied at y with left child x. Which subtree changes parent, and why is the in-order sequence unchanged?</summary>The right subtree of x (keys between x and y) becomes the left subtree of y; in-order before and after is (left of x), x, (that subtree), y, (right of y).</details>

<details><summary>6. A node has 8 malignant and 2 benign cases. Compute Gini and entropy. A split sends the 2 benign and 2 malignant left and 6 malignant right: compute the decrease in Gini.</summary>Gini \(= 1 - (0.64 + 0.04) = 0.32\); entropy \(\approx 0.722\) bits. Left: Gini 0.5; right: 0. Weighted \(0.4 \times 0.5 = 0.2\); decrease \(0.12\).</details>

<details><summary>7. Why do CART and C4.5 grow trees with Gini or entropy rather than with the misclassification error that is finally reported?</summary>Gini and entropy are strictly concave, so any split that makes the children purer decreases them even without changing a majority class; the piecewise-linear misclassification error can remain unchanged under a useful split and would stop growth prematurely.</details>

<details><summary>8. In Section 6.4 the stump on cell size and the exhaustive scan agreed, but the stump on all nine features chose cell shape. What does this say about the stability of the root split?</summary>The choice between near-duplicate features is decided by a small difference in weighted Gini, hence by a few training cases. A bootstrap sample can flip it, and because every lower split depends on the root the whole tree can change — the instability bagging exploits.</details>

<details><summary>9. Code reading: a colleague selects the tree depth from the orange test-accuracy curve of Figure 9 instead of the blue one. What is wrong?</summary>The test set has been used for model selection, so its accuracy at the chosen depth is optimistically biased. Choose depth by cross-validation (or OOB) on the training part and evaluate the test set once, at the end.</details>

<details><summary>10. Explain in one sentence each what bagging and random feature subsets do to \(\rho\sigma^2 + (1-\rho)\sigma^2/B\).</summary>Bagging increases \(B\), shrinking the second term; random feature subsets decrease \(\rho\) by making trees choose different splits, lowering the floor, at the cost of a somewhat larger \(\sigma^2\) per tree.</details>

<details><summary>11. With 478 training cases and 300 trees, roughly how many trees have a given case out of bag, and why does the OOB estimate need no separate test set?</summary>About \(0.368 \times 300 \approx 110\). Those trees never saw the case, so their vote is a prediction on unseen data; aggregated over all cases, the votes estimate the generalisation error.</details>

<details><summary>12. In Figure 15 cell shape has the highest impurity importance, yet permuting it costs the forest little. Give the mechanism, and what you would tell a clinician who asks which measurement matters most.</summary>Impurity importance credits the feature chosen at many splits; permutation measures the loss when it is scrambled while the correlated cell size, which carries the same information, remains available. Report the correlated size-and-shape measurements as a group, say that the ranking within it is arbitrary, and show partial-dependence curves rather than a single number.</details>

## Glossary

<dl class="gloss">
<dt>AVL tree</dt><dd>Self-balancing BST in which the subtree heights at every node differ by at most one; height at most about \(1.44\log_2 n\).</dd>
<dt>Bagging</dt><dd>Bootstrap aggregating: training predictors on bootstrap samples and combining them by vote or average to reduce variance.</dd>
<dt>Binary search tree (BST)</dt><dd>Binary tree of keys in which, at every node, all left keys are smaller and all right keys larger than the node's key.</dd>
<dt>Binary tree</dt><dd>Rooted tree with at most two children per node, distinguished as left and right; recursively, empty or a root with two binary subtrees.</dd>
<dt>Bootstrap sample</dt><dd>\(n\) cases drawn with replacement from \(n\); omits about 36.8 % of the distinct cases.</dd>
<dt>B-tree</dt><dd>Balanced search tree with many keys per node, designed for disk; the basis of database indexes.</dd>
<dt>CART</dt><dd>Classification and Regression Trees: greedy recursive partitioning with Gini impurity and cost-complexity pruning; the algorithm of the <code>rpart</code> package.</dd>
<dt>Cost-complexity pruning</dt><dd>Choosing the subtree minimising training error plus \(\alpha\) (<code>rpart</code>'s <code>cp</code>) times the number of leaves, with \(\alpha\) selected by cross-validation.</dd>
<dt>Decision tree</dt><dd>Binary tree whose internal nodes test a feature against a threshold and whose leaves hold a class or value; its decision regions are axis-aligned boxes.</dd>
<dt>Depth / height</dt><dd>Depth of a node: edges from the root. Height: edges on the longest downward path to a leaf; the tree's height is its root's (−1 if empty).</dd>
<dt>Entropy / Gini impurity</dt><dd>\(-\sum_k p_k\log_2 p_k\) (ID3/C4.5) and \(1 - \sum_k p_k^2\) (CART): the impurity measures whose decrease selects a split.</dd>
<dt>Gradient boosting</dt><dd>Ensemble that adds shallow trees sequentially, each fitted to the gradient of the loss so far; reduces bias.</dd>
<dt>Impurity-based (MDI) importance</dt><dd>Total impurity decrease attributable to a feature; biased towards features with many split points and arbitrary among correlated features.</dd>
<dt>Out-of-bag (OOB) error</dt><dd>Error of a forest on each training case as predicted by the trees whose bootstrap sample excluded it.</dd>
<dt>Partial dependence</dt><dd>Average prediction as a function of one feature with the others held at their observed values.</dd>
<dt>Permutation importance</dt><dd>Decrease in performance when one feature's values are randomly permuted.</dd>
<dt>Random forest</dt><dd>Bagged unpruned trees in which each split considers a random subset of the features.</dd>
<dt>Red–black tree</dt><dd>Self-balancing BST with a colour bit per node bounding the height by \(2\log_2(n+1)\).</dd>
<dt>Rotation</dt><dd>Local rearrangement of a node and one child that changes subtree heights by one while preserving the in-order sequence.</dd>
<dt>Traversal</dt><dd>Visiting every node once: pre-order (root first), in-order (root between subtrees), post-order (root last), level-order (by depth).</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Knuth DE. <em>The Art of Computer Programming, Volume 1: Fundamental Algorithms</em>. 3rd ed. Addison-Wesley; 1997. Section 2.3, Trees. No DOI; publisher page: <a href="https://www.informit.com/store/art-of-computer-programming-volume-1-fundamental-algorithms-9780201896831">informit.com</a></li>
<li id="r2">Knuth DE. <em>The Art of Computer Programming, Volume 3: Sorting and Searching</em>. 2nd ed. Addison-Wesley; 1998. Section 6.2, Searching by comparison of keys (binary search trees, balanced trees). No DOI; publisher page: <a href="https://www.informit.com/store/art-of-computer-programming-volume-3-sorting-and-searching-9780201896855">informit.com</a></li>
<li id="r3">Adelson-Velskii GM, Landis EM. An algorithm for the organization of information. <em>Doklady Akademii Nauk SSSR</em>. 1962;146(2):263–266 (in Russian; English translation in <em>Soviet Mathematics Doklady</em>. 1962;3:1259–1263). No DOI; original on Math-Net.Ru: <a href="https://www.mathnet.ru/eng/dan26964">mathnet.ru/eng/dan26964</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">Guibas LJ, Sedgewick R. A dichromatic framework for balanced trees. In: <em>Proceedings of the 19th Annual Symposium on Foundations of Computer Science (SFCS 1978)</em>. IEEE; 1978:8–21. <a href="https://doi.org/10.1109/SFCS.1978.3">doi:10.1109/SFCS.1978.3</a></li>
<li id="r6">Bayer R, McCreight E. Organization and maintenance of large ordered indexes. <em>Acta Informatica</em>. 1972;1(3):173–189. <a href="https://doi.org/10.1007/BF00288683">doi:10.1007/BF00288683</a></li>
<li id="r7">Breiman L, Friedman JH, Olshen RA, Stone CJ. <em>Classification and Regression Trees</em>. Wadsworth; 1984. Reissued by Chapman and Hall/CRC; 2017. <a href="https://doi.org/10.1201/9781315139470">doi:10.1201/9781315139470</a></li>
<li id="r8">Quinlan JR. Induction of decision trees. <em>Machine Learning</em>. 1986;1(1):81–106. <a href="https://doi.org/10.1007/BF00116251">doi:10.1007/BF00116251</a></li>
<li id="r9">Quinlan JR. <em>C4.5: Programs for Machine Learning</em>. Morgan Kaufmann; 1993. The book has no DOI; see the review by Salzberg SL. <em>Machine Learning</em>. 1994;16(3):235–240. <a href="https://doi.org/10.1007/BF00993309">doi:10.1007/BF00993309</a></li>
<li id="r10">Hyafil L, Rivest RL. Constructing optimal binary decision trees is NP-complete. <em>Information Processing Letters</em>. 1976;5(1):15–17. <a href="https://doi.org/10.1016/0020-0190(76)90095-8">doi:10.1016/0020-0190(76)90095-8</a></li>
<li id="r11">Breiman L. Bagging predictors. <em>Machine Learning</em>. 1996;24(2):123–140. <a href="https://doi.org/10.1007/BF00058655">doi:10.1007/BF00058655</a></li>
<li id="r12">Breiman L. Random forests. <em>Machine Learning</em>. 2001;45(1):5–32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r13">Ho TK. Random decision forests. In: <em>Proceedings of the 3rd International Conference on Document Analysis and Recognition</em>. IEEE; 1995;1:278–282. <a href="https://doi.org/10.1109/ICDAR.1995.598994">doi:10.1109/ICDAR.1995.598994</a></li>
<li id="r14">Ho TK. The random subspace method for constructing decision forests. <em>IEEE Transactions on Pattern Analysis and Machine Intelligence</em>. 1998;20(8):832–844. <a href="https://doi.org/10.1109/34.709601">doi:10.1109/34.709601</a></li>
<li id="r15">Breiman L. <em>Out-of-bag estimation</em>. Technical report, Statistics Department, University of California, Berkeley; 1996. No DOI; <a href="https://www.stat.berkeley.edu/~breiman/OOBestimation.pdf">stat.berkeley.edu/~breiman/OOBestimation.pdf</a></li>
<li id="r16">Strobl C, Boulesteix A-L, Zeileis A, Hothorn T. Bias in random forest variable importance measures: illustrations, sources and a solution. <em>BMC Bioinformatics</em>. 2007;8:25. <a href="https://doi.org/10.1186/1471-2105-8-25">doi:10.1186/1471-2105-8-25</a></li>
<li id="r17">Strobl C, Boulesteix A-L, Kneib T, Augustin T, Zeileis A. Conditional variable importance for random forests. <em>BMC Bioinformatics</em>. 2008;9:307. <a href="https://doi.org/10.1186/1471-2105-9-307">doi:10.1186/1471-2105-9-307</a></li>
<li id="r18">Altmann A, Toloşi L, Sander O, Lengauer T. Permutation importance: a corrected feature importance measure. <em>Bioinformatics</em>. 2010;26(10):1340–1347. <a href="https://doi.org/10.1093/bioinformatics/btq134">doi:10.1093/bioinformatics/btq134</a></li>
<li id="r19">Friedman JH. Greedy function approximation: a gradient boosting machine. <em>Annals of Statistics</em>. 2001;29(5):1189–1232. <a href="https://doi.org/10.1214/aos/1013203451">doi:10.1214/aos/1013203451</a></li>
<li id="r20">Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: <em>Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining</em>. ACM; 2016:785–794. <a href="https://doi.org/10.1145/2939672.2939785">doi:10.1145/2939672.2939785</a></li>
<li id="r21">Grinsztajn L, Oyallon E, Varoquaux G. Why do tree-based models still outperform deep learning on typical tabular data? <em>Advances in Neural Information Processing Systems</em> 35 (NeurIPS 2022, Datasets and Benchmarks track). 2022:507–520. arXiv:2207.08815. <a href="https://doi.org/10.48550/arXiv.2207.08815">doi:10.48550/arXiv.2207.08815</a></li>
<li id="r22">Fernández-Delgado M, Cernadas E, Barro S, Amorim D. Do we need hundreds of classifiers to solve real world classification problems? <em>Journal of Machine Learning Research</em>. 2014;15(90):3133–3181. No DOI; <a href="https://jmlr.org/papers/v15/delgado14a.html">jmlr.org/papers/v15/delgado14a.html</a></li>
<li id="r23">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning: Data Mining, Inference, and Prediction</em>. 2nd ed. Springer; 2009. Chapters 9.2 (trees), 10 (boosting) and 15 (random forests). <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r24">Louppe G. <em>Understanding Random Forests: From Theory to Practice</em>. PhD thesis, University of Liège; 2014. arXiv:1407.7502. <a href="https://doi.org/10.48550/arXiv.1407.7502">doi:10.48550/arXiv.1407.7502</a></li>
<li id="r25">Díaz-Uriarte R, Alvarez de Andrés S. Gene selection and classification of microarray data using random forest. <em>BMC Bioinformatics</em>. 2006;7:3. <a href="https://doi.org/10.1186/1471-2105-7-3">doi:10.1186/1471-2105-7-3</a></li>
<li id="r26">Stiell IG, Greenberg GH, McKnight RD, Nair RC, McDowell I, Worthington JR. A study to develop clinical decision rules for the use of radiography in acute ankle injuries. <em>Annals of Emergency Medicine</em>. 1992;21(4):384–390. <a href="https://doi.org/10.1016/S0196-0644(05)82656-3">doi:10.1016/S0196-0644(05)82656-3</a></li>
<li id="r27">Goldman L, Weinberg M, Weisberg M, Olshen R, Cook EF, Sargent RK, et al. A computer-derived protocol to aid in the diagnosis of emergency room patients with acute chest pain. <em>New England Journal of Medicine</em>. 1982;307(10):588–596. <a href="https://doi.org/10.1056/NEJM198209023071004">doi:10.1056/NEJM198209023071004</a></li>
<li id="r28">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proceedings of SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r29">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Annals of Statistics</em>. 2004;32(2):407–499 (source of the diabetes dataset used in the Python edition). <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r30">Breiman L. Statistical modeling: the two cultures (with comments and a rejoinder by the author). <em>Statistical Science</em>. 2001;16(3):199–231. <a href="https://doi.org/10.1214/ss/1009213726">doi:10.1214/ss/1009213726</a></li>
<li id="r31">Lundberg SM, Erion G, Chen H, DeGrave A, Prutkin JM, Nair B, et al. From local explanations to global understanding with explainable AI for trees. <em>Nature Machine Intelligence</em>. 2020;2:56–67. <a href="https://doi.org/10.1038/s42256-019-0138-9">doi:10.1038/s42256-019-0138-9</a></li>
<li id="r32">Shwartz-Ziv R, Armon A. Tabular data: deep learning is not all you need. <em>Information Fusion</em>. 2022;81:84–90. <a href="https://doi.org/10.1016/j.inffus.2021.11.011">doi:10.1016/j.inffus.2021.11.011</a></li>
<li id="r33">Harrison D, Rubinfeld DL. Hedonic housing prices and the demand for clean air. <em>Journal of Environmental Economics and Management</em>. 1978;5(1):81–102 (source of the Boston housing data). <a href="https://doi.org/10.1016/0095-0696(78)90006-2">doi:10.1016/0095-0696(78)90006-2</a></li>
<li id="r34">Therneau TM, Atkinson EJ. <em>An Introduction to Recursive Partitioning Using the RPART Routines</em>. Technical report, Mayo Foundation; rpart package vignette. No DOI; <a href="https://cran.r-project.org/package=rpart">cran.r-project.org/package=rpart</a></li>
<li id="r35">Liaw A, Wiener M. Classification and regression by randomForest. <em>R News</em>. 2002;2(3):18–22. No DOI; <a href="https://cran.r-project.org/doc/Rnews/">cran.r-project.org/doc/Rnews</a></li>
<li id="r36">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. Chapter 8, tree-based methods. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a> (free PDF at statlearning.com)</li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Cormen TH, Leiserson CE, Rivest RL, Stein C. <em>Introduction to Algorithms</em>. 4th ed. MIT Press; 2022. — Chapters 12 (binary search trees) and 13 (red–black trees) for a full treatment, with deletion and rotations, of Sections 2–4.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 9.2 for CART, Chapter 10 for boosting, Chapter 15 for random forests; the source of the variance formula of Section 7.1.<sup class="c"><a href="#r23">[23]</a></sup></li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Chapter 8, tree-based methods: the most accessible treatment of Sections 6–8, in R.<sup class="c"><a href="#r36">[36]</a></sup></li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. — the reference for <code>rpart</code>, <code>randomForest</code> and the other R modelling functions used here.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>Therneau TM, Atkinson EJ. <em>An Introduction to Recursive Partitioning Using the RPART Routines</em> — the <code>rpart</code> vignette, with the mathematics of CART and the complexity table of Section 6.7.<sup class="c"><a href="#r34">[34]</a></sup></li>
<li><code>randomForest</code> package documentation and Breiman &amp; Cutler's Random Forests pages — <a href="https://www.stat.berkeley.edu/~breiman/RandomForests/">stat.berkeley.edu/~breiman/RandomForests</a>; Breiman's own description of the algorithm, the OOB estimate and the importance measures.<sup class="c"><a href="#r35">[35]</a></sup></li>
<li>Interactive visualisations of binary search trees, AVL and red–black trees (University of San Francisco, D. Galles) — <a href="https://www.cs.usfca.edu/~galles/visualization/Algorithms.html">cs.usfca.edu/~galles/visualization</a>; useful to watch rotations happen.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; original source of the breast-cancer and housing data.</li>
</ul>
